# Devoir maison 1 — carnets J1 et J2

> **À rendre avant la première séance de la semaine suivante.** Ce devoir porte sur les carnets
> **J1** (du fichier brut à la soumission) et **J2** (ce que la table cache). Il se fait **seul**, à
> la maison, et il se rend.

**Ce qu'on rend :** ce carnet, rempli, déposé dans votre **dépôt GitHub personnel** avec la
déclaration d'usage de l'IA. **Les trois tableaux** de la fin et **l'outil de la question 12** sont
les pièces principales.

**Pourquoi ce devoir :** chaque question demande une mesure que **vous** produisez — jamais une
lecture. Presque toutes s'ouvrent sur **une prédiction écrite avant la mesure** : c'est le geste du
métier, et c'est aussi ce qui rend une erreur visible.

**Quatre mots, pour être sûr :**

- **les deux réponses** — le patient vient (`patient_absent` vaut 0), ou il ne vient pas (1) ;
- **la référence** — le score d'un modèle qui répond toujours la même chose : le minimum à battre ;
- **la barre de bruit** — de combien un score bouge tout seul quand seul le découpage change ;
  en dessous de **deux barres**, un écart n'est pas une différence ;
- **un constat** — une observation, un chiffre, **par rapport à quoi**, et ce que vous en faites.

**Quinze questions, dont douze de cœur.** Chaque titre porte sa mention : **cœur** = le socle du
semestre ; **approfondissement** = à faire dès que le cœur tient. Si le temps manque, faites les
douze de cœur d'abord — les trois approfondissements portent les gestes les plus fins, et ils se
reprennent plus tard. Rien n'est facultatif au sens où tout est lu.

---

## Comment ça marche

1. La partie **« Mise en place »** est **fournie** : exécutez-la de haut en bas. **Deux de ses
   cellules sont faites pour être touchées** : votre **identifiant GitHub** (les questions 2 et 11
   mesurent sur un échantillon tiré de lui), et `ajouter_features` (la question 10).
   **Un identifiant mal recopié change votre échantillon** : copiez-le comme dans l'URL de votre dépôt.
2. Chaque **question** porte sa consigne, en markdown, un **emplacement vide**, et des
   **contrôles de forme** posés dans la cellule : ils disent si votre réponse a la forme attendue
   (un pourcentage, une dimension, une cohérence avec **vos** autres réponses). Ils ne donnent
   **jamais** la valeur attendue, et ils se taisent tant que vous n'avez rien écrit.
3. Les questions sont dans l'ordre des carnets : **1 à 8 sur J1**, **9 à 15 sur J2** — sauf la 15,
   qui transpose la méthode hors de ce jeu de données.
4. Une réponse se **chiffre** ou s'**écrit**. Plusieurs questions demandent d'**annoncer AVANT de
   mesurer** : `noter(...)` refuse d'ailleurs d'enregistrer un essai non annoncé.
5. À la fin : les **trois tableaux**, puis le **dépôt**.

**Ce qui est lu.** **Les trois tableaux** — constats, hypothèses, essais — **et le code**, là où il
produit un chiffre. **Ce dépôt est relevé** : il est lu, et **les sept devoirs comptent ensemble
pour 10 % de la note**, dans la composante « exercices et devoirs » (25 %, avec E1, E2, E3).
**Il n'y a pas d'oral sur les devoirs** : l'oral individuel, en fin de parcours, porte sur
l'ensemble de votre travail — les trois blocs et les deux terrains.

> **La règle, la même qu'en séance :** un score sans référence ne veut rien dire, un écart plus
> petit que deux barres n'est pas une différence, et un constat sans chiffre n'en est pas un.


## Mise en place (fournie)

Ces cellules sont **extraites de `notebook_J1.ipynb` et de `notebook_J2.ipynb`**, qui restent les
sources uniques de la table de travail, de la référence et du modèle. Exécutez-les de haut en bas,
sans rien y modifier — sauf `ajouter_features`, que la question 10 vous demandera d'écrire.

Les `random_state=0` que vous verrez sont la **graine du tirage au sort** (expliquée en J1 §5) :
le même nombre redonne le même découpage, donc la même mesure.


### Les outils


In [1]:
# les outils
# Aucun import en bloc au début : chaque module entre dans le carnet au moment de son premier
# usage. On voit ainsi à quoi il sert, au lieu de le deviner dans une liste.

# `display` : un tableau s'affiche proprement dans un carnet, et reste lisible ailleurs.
# C'est par `display` qu'on montre TOUTES les tables de ce carnet — jamais un print de texte.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


### Les données


In [2]:
import os
import pandas as pd
# où sont les fichiers — on CHERCHE `train.csv`, on ne suppose pas un chemin
# (Kaggle a déjà déplacé ses points de montage : un chemin recopié se périme, une recherche non)
def _trouver_dossier(racines):
    """Première racine contenant train.csv — en surface, puis en profondeur."""
    for racine in racines:
        if not os.path.isdir(racine):
            continue
        if os.path.exists(os.path.join(racine, "train.csv")):
            return racine
        for chemin, _, fichiers in os.walk(racine):
            if "train.csv" in fichiers:
                return chemin
    return None

DOSSIER = _trouver_dossier(["/kaggle/input", "sortie"])
assert DOSSIER, "train.csv introuvable : les données de la compétition sont-elles montées ?"


In [3]:
train = pd.read_csv(os.path.join(DOSSIER, "train.csv"), low_memory=False)
test = pd.read_csv(os.path.join(DOSSIER, "test.csv"), low_memory=False)

ID, CIBLE = "ligne_id", "patient_absent"


### La table de travail


In [4]:
A_ECARTER = [ID, "dossier_id", "patient_id", "duree_consultation", "constantes_relevees",
             "motif_absence", "date_prise", "date_rdv", "taille_cm", "date_naissance",
             "canal_prise"]

def matrice(cols):
    """La table de travail : colonnes écartées retirées, catégorielles déclarées."""
    X = cols.drop(columns=[c for c in A_ECARTER + [CIBLE] if c in cols.columns]).copy()
    cat = [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]
    trop_fin = [c for c in cat if X[c].nunique() > 255]
    X = X.drop(columns=trop_fin)
    cat = [c for c in cat if c not in trop_fin]
    for c in cat:
        X[c] = X[c].fillna("").astype("category")
    return X, cat, trop_fin

X, CAT, TROP_FIN = matrice(train)
Xt, _, _ = matrice(test)

for c in CAT:
    Xt[c] = Xt[c].cat.set_categories(X[c].cat.categories)


### La cible, et la référence


In [5]:
repartition = pd.DataFrame({"lignes": [len(train), int(train[CIBLE].sum())],
                            "part (%)": [100.0, round(100 * train[CIBLE].mean(), 2)]},
                           index=["toutes", "absents"])

reference = 100 * max(train[CIBLE].mean(), 1 - train[CIBLE].mean())


In [6]:
y = train[CIBLE]
reference = 100 * max(y.mean(), 1 - y.mean())


### Le premier modèle


In [7]:
# le modèle et le découpage : ils entrent ici
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
Xtr, Xval, ytr, yval = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

modele = HistGradientBoostingClassifier(random_state=0).fit(Xtr, ytr)
score_interne = 100 * modele.score(Xval, yval)


### Vos features — LA cellule que vous modifiez


In [8]:
# QUESTION 10 — Feature personnelle.
# Le constat de la question 3 montre que solde_credit comporte des valeurs manquantes.
# On ajoute un indicateur binaire : 1 si le solde manque, 0 s'il est renseigné.
# La fonction est appliquée de la même façon à l'entraînement et aux données à prédire.
def ajouter_features(df):
    df = df.copy()
    if "solde_credit" in df.columns:
        df["solde_credit_manquant"] = df["solde_credit"].isna().astype("int8")
    return df

# Important : ne créez pas cette colonne à la main ailleurs.
# La chaîne preparer() et remesurer() doivent appliquer la même transformation des deux côtés.


### La chaîne — une seule, des deux côtés


In [9]:
# ── LA CHAÎNE : une seule fonction, celle que `predict` ET l'entraînement utilisent.
def preparer(brut):
    """Écarte, type, applique vos features : LA chaîne, des deux côtés.
       `predict` l'appelle ; l'entraînement l'appelle aussi, par `matrice()` sur la table de
       travail. Contourner cette fonction, c'est apprendre sur une table et prédire sur une
       autre — exactement le défaut que la séance 2 traque."""
    d = brut.drop(columns=[c for c in A_ECARTER + TROP_FIN + [CIBLE] if c in brut.columns],
                  errors="ignore").copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].fillna("").astype("category").cat.set_categories(X[c].cat.categories)
    return ajouter_features(d)

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis X : ré-exécutable sans que la
       feature se superpose à elle-même."""
    global modele
    Xc = ajouter_features(X)
    a, b, c, d = train_test_split(Xc, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)


### L'interface — le modèle qu'on peut appeler


In [10]:
# numpy, pour la seule assertion qui suit
import numpy as np

def predict(df):
    """Renvoie un tableau de 0/1, un par ligne de df. C'est cette fonction que le cours appellera.
       Elle passe par `preparer(...)`, la même chaîne que l'entraînement — pas une copie."""
    # elle ne refait PAS la chaîne à la main : elle appelle `preparer`, celle de l'entraînement.
    # C'est ce couplage qui garantit qu'une table n'est jamais préparée de deux façons.
    return modele.predict(preparer(df)).astype(int)

def test_predict():
    """Vérifie que l'interface répond ce qu'on attend d'elle."""
    p = predict(test.head(500))

    assert len(p) == 500, f"attendu 500 prédictions, obtenu {len(p)}"
    assert set(np.unique(p)) <= {0, 1}, "les valeurs doivent être 0 ou 1"
    print(f"test OK — 500 prédictions, {int(sum(p))} positifs")

# test_predict()   (au devoir : la cellule « l'interface » s'en charge)


### Le tableau d'essais


In [11]:
# l'horloge entre ici : c'est elle qui rend une prédiction vérifiable
from datetime import datetime

ESSAIS = []
PREDICTIONS = []

def prevoir(essai, attendu, raison):
    """Écrit ce que vous attendez AVANT de mesurer — horodaté.
       `noter(...)` refuse ensuite tout essai sans prédiction : l'ordre n'est pas demandé, il est
       appliqué. Ce que cette fonction garantit, exactement : une prédiction EXISTAIT avant que la
       ligne du tableau soit écrite. Elle ne date pas la mesure — le score lui arrive déjà calculé.
       Pour la garantie complète, mesurez dans `essai(...)`, qui date la mesure lui-même."""
    PREDICTIONS[:] = [p for p in PREDICTIONS if p["essai"] != essai]
    PREDICTIONS.append({"essai": essai, "attendu": attendu, "raison": raison,
                        "écrit à": datetime.now().strftime("%H:%M:%S")})
    display(pd.DataFrame(PREDICTIONS))

def essai(nom, mesure, changement="", attendu=""):
    """Le chemin STRICT : la mesure est prise ICI, donc l'ordre (prédiction puis mesure) est prouvé.
       `mesure` est une fonction sans argument qui rend le score, par exemple
       `lambda: modele.score(Xval, yval)`. Refuse de mesurer sans prédiction."""
    prevu = [p for p in PREDICTIONS if p["essai"] == nom]
    if not prevu:
        print(f"refusé : aucune prédiction pour « {nom} ». Écrivez prevoir(...) d'abord.")
        return None
    score = 100 * mesure()
    noter(nom, changement or "voir la question", attendu or prevu[0]["attendu"], score)
    return score

def noter(hypothese, changement, attendu, score):
    """Ajoute une ligne au tableau. `attendu` : un signe et une raison, jamais une valeur.
       Ré-exécuter une cellule REMPLACE la ligne de même `hypothèse` : elle ne la double pas.

       REFUSE d'enregistrer un essai non annoncé — c'est la règle du cours, appliquée.
       Ce qu'elle garantit, et rien de plus : une prédiction existait AVANT que cette ligne soit
       écrite. Elle ne date pas la mesure — d'où `essai(...)` pour la garantie complète."""
    prevu = [p for p in PREDICTIONS if p["essai"] == hypothese]
    if not prevu:
        print(f"  refusé : aucune prédiction pour « {hypothese} ».")
        print("  Écrivez prevoir(...) d'abord — l'ordre est vérifié, pas seulement demandé.")
        return
    ligne = {"hypothèse": hypothese, "changement": changement,
             "attendu": attendu, "mesuré": round(score, 2),
             "prédiction écrite avant ?": "oui, à " + prevu[0]["écrit à"]}
    ESSAIS[:] = [e for e in ESSAIS if e["hypothèse"] != hypothese] + [ligne]
    display(pd.DataFrame(ESSAIS))

# les deux amorces du carnet sont annoncées elles aussi — sinon elles ne passeraient pas
prevoir("rien faire du tout", "c'est le minimum à battre",
        "un modèle qui répond toujours la même chose vaut la classe majoritaire")
prevoir("un modèle suffit-il ?", "mieux que la référence",
        "les arbres captent une partie du signal, pas tout")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26


### La soumission


In [12]:
soumission = pd.DataFrame({ID: test[ID], CIBLE: modele.predict(Xt).astype(int)})
soumission.to_csv("submission.csv", index=False)


### Le modèle linéaire


In [13]:
# le modèle linéaire : il entre ici
from sklearn.linear_model import LogisticRegression
X_num = pd.get_dummies(X, columns=CAT).fillna(0)
Xtr_n, Xval_n = X_num.loc[Xtr.index], X_num.loc[Xval.index]

modele_lineaire = LogisticRegression(solver="newton-cholesky").fit(Xtr_n, ytr)
score_lineaire = 100 * modele_lineaire.score(Xval_n, yval)


### Les outils du devoir : les tableaux, la barre, les contrôles


In [14]:
# les trois tableaux, et les outils qui les remplissent

CONSTATS = []

def constat(observation, chiffre, comparaison, consequence):
    """observation : ce qu'on a vu · chiffre : la mesure exacte
       comparaison : par rapport à quoi ? (un nombre) · conséquence : garder, écarter,
       surveiller, mesurer. Ré-exécuter une cellule REMPLACE la ligne de même `observation`."""
    ligne = {"observation": observation, "chiffre": chiffre,
             "comparaison": comparaison, "conséquence": consequence}
    CONSTATS[:] = [c for c in CONSTATS if c["observation"] != observation] + [ligne]
    display(pd.DataFrame(CONSTATS))

HYPOTHESES = []

def hypothese(feature, ce_qu_elle_raconte, comment_on_la_test, ce_qui_la_contredirait):
    """Une hypothèse est une prédiction qu'on peut tester, avec un critère annoncé AVANT.
       Ré-exécuter une cellule REMPLACE la ligne de même `feature`."""
    ligne = {"feature": feature, "ce qu'elle raconte": ce_qu_elle_raconte,
             "test": comment_on_la_test, "contredite si": ce_qui_la_contredirait}
    HYPOTHESES[:] = [h for h in HYPOTHESES if h["feature"] != feature] + [ligne]
    display(pd.DataFrame(HYPOTHESES))

# les trois constats de la séance 1 — ils comptent : la question 3 en demande CINQ au total,
# donc DEUX des vôtres. Ils sont réécrits ici parce que ce carnet doit tourner seul.
constat("une colonne ne dit rien",
        f"{train['zone_clinique'].nunique()} valeur(s) distincte(s)",
        f"sur les {len(train)} lignes de la table, soit une colonne constante au sens du cours",
        "l'écarter — et ce qu'on y gagne se mesure")
constat("deux colonnes disent la même chose",
        "corrélation 1,000 entre cout_acte et montant_acte",
        f"contre moins de 0,05 pour les autres paires sur {len(train)} lignes : un rapport constant",
        "n'en garder qu'une")
constat("la cible est déséquilibrée",
        f"{100 * train[CIBLE].mean():.2f} % d'absents",
        "contre 50 % pour un problème équilibré : la référence monte d'autant",
        "mesurer : cela conditionne la métrique, donc ce qu'on regarde")
CONSTATS_DE_LA_SEANCE = [c['observation'] for c in CONSTATS]

# les deux amorces de J1 — annoncées, puis notées (c'est la règle de l'ordre, montrée)
noter("rien faire du tout", "classe majoritaire", "c'est le minimum à battre", reference)
noter("un modèle suffit-il ?", "HistGradientBoosting", "mieux que la référence", score_interne)

# la barre de bruit — recalculée ici parce que la question 5 en change la taille du jeu
erreur_type = 100 * ((score_interne / 100) * (1 - score_interne / 100) / len(yval)) ** 0.5
print(f"erreur-type de l'exactitude de validation (jeu de 20 %) : ± {erreur_type:.2f} point(s)")
print("  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.")
print("  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.")

# ── VOTRE échantillon : il dépend de VOTRE identifiant GitHub, et de lui seul.
#    Les questions 2 et 11 mesurent dessus : deux voisins n'ont donc pas les mêmes chiffres,
#    et un chiffre recopié ne tombe pas juste. Le reste du devoir (le socle J1/J2) garde le
#    protocole figé du cours — c'est voulu : ce qui doit être comparable entre tous reste
#    comparable, et ce qui doit être VOTRE mesure est la vôtre.
IDENTIFIANT_GITHUB = "marcedoun-byte"          # <- remplacez par votre pseudo GitHub, comme dans l'URL
import hashlib
if IDENTIFIANT_GITHUB.strip() and IDENTIFIANT_GITHUB.strip() != '…':
    GRAINE = int(hashlib.sha256(IDENTIFIANT_GITHUB.strip().lower().encode('utf-8')).hexdigest()[:8], 16)
else:
    GRAINE = None
mon_train = train.sample(frac=0.6, random_state=GRAINE) if GRAINE is not None else train
print(f"votre échantillon : {len(mon_train)} lignes sur {len(train)}"
      + ("" if GRAINE is not None else "   (identifiant non renseigné : la table entière)"))

# la table de travail NUE, gardée de côté : c'est sur elle que `remesurer()` rebâtit `X`,
# pour que votre feature s'applique UNE SEULE fois, même si vous relancez la cellule dix fois.
X_sans_features = matrice(train)[0]

def remesurer():
    """Réentraîne sur la table de travail COURANTE et met `modele` à jour — et `X` avec.
       `ajouter_features` n'est appliquée qu'UNE fois, depuis la table nue : ré-exécutable."""
    global modele, X
    X = ajouter_features(X_sans_features)
    a, b, c, d = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
    modele = HistGradientBoostingClassifier(random_state=0).fit(a, c)
    return 100 * modele.score(b, d)

VERIFICATIONS = []

def verifier(nom, condition, indice=''):
    """Contrôle silencieux : il ne dit JAMAIS la valeur attendue."""
    VERIFICATIONS.append({'contrôle': nom, 'tient': bool(condition)})
    if condition:
        print('  ok        ' + nom)
    else:
        print('  À REVOIR  ' + nom + '   ->   ' + indice)

def controle(nom, quoi, indice='', des_que=None):
    """Contrôle de FORME, posé dans la cellule de la question. Il ne compare JAMAIS à une valeur
       attendue : un type, une plage, une dimension, ou une cohérence avec VOS autres réponses.
       Tant que la réponse n'est pas écrite, il se tait.

       `des_que` : une fonction sans argument qui dit si la réponse EXISTE déjà. Tant qu'elle rend
       False, le contrôle se tait — un contrôle parle d'une réponse écrite, jamais de son absence."""
    if des_que is not None:
        try:
            if not des_que():
                print(f'  (pas encore écrit)  {nom}')
                return
        except NameError:
            print(f'  (pas encore écrit)  {nom}')
            return
    try:
        ok = bool(quoi())
    except NameError:
        print(f'  (pas encore écrit)  {nom}')
        return
    except Exception as e:
        print(f'  REVOIR    {nom}   ->   {type(e).__name__} : {e}')
        return
    VERIFICATIONS.append({'contrôle': nom, 'tient': ok})
    print(('  ok        ' if ok else '  REVOIR    ') + nom + ('' if ok else '   ->   ' + indice))

# le nombre de colonnes de la table AVANT votre feature (la question 10 s'y compare)
n_colonnes_avant = X.shape[1]

# les deux essais du haut du carnet J1 sont des AMORCES : ils ne comptent pas comme les vôtres.
AMORCES = ('rien faire du tout', 'un modèle suffit-il ?')

# la clé de la question 15 vit côté enseignant, pas ici : un vocabulaire de quatre familles,
# c'est 4^6 combinaisons, et une empreinte de nom de colonne se casse en quelques secondes.


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"


erreur-type de l'exactitude de validation (jeu de 20 %) : ± 0.22 point(s)
  formule : racine de p(1-p)/n, où n est le nombre de lignes de validation.
  vocabulaire : une barre de bruit EST une erreur-type — « deux barres » = deux erreurs-types.
votre échantillon : 89936 lignes sur 149894


### L'interface, telle qu'on l'appelle


In [15]:
# `predict` est définie par la mise en place, mais elle est testée ICI : ainsi, `Run All`
# fonctionne même après que vous avez écrit votre feature (la Q9 de l'ancien devoir cassait là).
# Relancez CETTE cellule après avoir touché `ajouter_features` : `X` reçoit la feature une seule
# fois, `modele` est mis à jour, et `predict` l'applique aussi de son côté.
print(f"exactitude de validation, table de travail courante : {remesurer():.2f} %")
test_predict()

import matplotlib.pyplot as plt          # pour les graphiques des questions 2 et 13


exactitude de validation, table de travail courante : 82.51 %
test OK — 500 prédictions, 31 positifs


## Les questions

**Quinze questions : douze de cœur, trois d'approfondissement.** Les huit premières portent sur
**J1**, les sept dernières sur **J2**. Pour chacune : lisez la consigne, écrivez votre code (ou
votre phrase) à l'emplacement prévu, et **exécutez la cellule** — c'est l'exécution qui déclenche
les contrôles.


### Question 1 — La référence, recalculée et prouvée — et le collègue qui annonce 83 %  ·  *carnet J1*  ·  *cœur*


La référence est le chiffre qui décide si tout ce qui suit vaut quelque chose : le score
d'un modèle qui répond **toujours la même chose**. Le carnet l'a calculée sur **tout** `train`.

À faire :
  1. calculez la part d'absents **sur les lignes de contrôle** (`yval`), en **une** ligne, sans
     réutiliser `reference` → `part_absents_controle` ;
  2. **deux chiffres différents, et c'est le piège de la séance** : cette part d'absents n'est **pas**
     la référence. La référence est une **exactitude** — la part de lignes bien classées par un
     modèle qui répond toujours la classe majoritaire (ici : « le patient vient ») : calculez-la
     **sur les lignes de contrôle** → `ma_reference` ;
  3. construisez ce prédicteur : un **tableau de 0/1 de la même longueur que `yval`**, rempli de la
     classe majoritaire — puis montrez qu'il **atteint** `ma_reference` — pas « environ » ;
  4. écrivez l'écart entre `ma_reference` et `reference` (celle de la mise en place, calculée sur
     tout `train`) → `ecart_a_train`, et **pourquoi ce petit écart compte** quand on compare deux
     modèles ;
  5. **un collègue annonce 83 %** d'exactitude. Écrivez votre réponse en **deux chiffres** :
     l'écart à la référence, puis **le nombre de barres de bruit** que vaut cet écart. Un
     score qui ne bat pas la référence de deux barres n'est pas un résultat.

À écrire : `part_absents_controle`, `ma_reference`, `ecart_a_train`, `reponse_au_collegue`,
`ecart_en_barres`.


In [16]:
# QUESTION 1 — Référence sur le jeu de contrôle.
# Le taux d'absents est une proportion de 1 ; la référence est l'exactitude de la classe majoritaire.
part_absents_controle = 100 * yval.mean()
ma_reference = 100 * max(yval.mean(), 1 - yval.mean())

# Un modèle de référence prédit toujours la classe la plus fréquente du jeu d'entraînement.
classe_majoritaire = int(ytr.mode().iloc[0])
pred = np.full(len(yval), classe_majoritaire, dtype=int)
score_reference_controle = 100 * np.mean(pred == yval.to_numpy())

# On compare la référence recalculée sur la validation à celle calculée sur tout train.
ecart_a_train = ma_reference - reference

# Le collègue annonce 83 %. On exprime son avance sur la référence et en barres de bruit.
ecart_collegue = 83 - ma_reference
ecart_en_barres = ecart_collegue / erreur_type
reponse_au_collegue = (
    f"Le score de 83 % est à {ecart_collegue:.2f} point(s) de la référence "
    f"de {ma_reference:.2f} %, soit {ecart_en_barres:.2f} barre(s) de bruit. "
    f"Il faut dépasser 2 barres pour parler d'un écart suffisamment net. "
    f"Le prédicteur majoritaire obtient {score_reference_controle:.2f} %."
)

print(f"Part d'absents dans yval : {part_absents_controle:.2f} %")
print(f"Référence sur la validation : {ma_reference:.2f} %")
print(f"Score du prédicteur majoritaire : {score_reference_controle:.2f} %")
print(f"Écart avec la référence de train : {ecart_a_train:.4f} point(s)")
print(reponse_au_collegue)


Part d'absents dans yval : 18.25 %
Référence sur la validation : 81.75 %
Score du prédicteur majoritaire : 81.75 %
Écart avec la référence de train : 0.0015 point(s)
Le score de 83 % est à 1.25 point(s) de la référence de 81.75 %, soit 5.69 barre(s) de bruit. Il faut dépasser 2 barres pour parler d'un écart suffisamment net. Le prédicteur majoritaire obtient 81.75 %.


### Question 2 — Le balayage, le piège des valeurs rares, et le critère des 500 lignes  ·  *carnet J1*  ·  *cœur*


Trouvez, en regroupant, la colonne qui **sépare le mieux** les deux réponses. Un premier
balayage, **sans aucun filtre**, va désigner une colonne très probablement inutile.

**Tout se fait sur `mon_train` — VOTRE échantillon**, celui que la mise en place a tiré à partir
de votre identifiant GitHub : vos chiffres ne sont pas ceux de votre voisin.

À faire :
  1. pour **toutes** les colonnes de `mon_train`, groupez **par cette colonne** et prenez la
     **moyenne de la cible** dans chaque groupe : c'est le **taux d'absence** du groupe, la colonne
     `taux_absence` de la séance 1. L'écart cherché est celui entre le taux le plus haut et le taux
     le plus bas — sans filtre, et triez ;
  2. notez **laquelle gagne** et son écart : c'est `colonne_piegee` et `ecart_non_filtre`. Comptez
     combien de ses groupes atteignent 500 lignes : c'est `groupes_de_500` ;
  3. **puis** appliquez le critère du cours : **ne gardez que les groupes d'au moins 500 lignes, et
     comparez ceux-là** — refaites le classement (`classement`), et notez le plus petit groupe gardé
     (`plus_petit_groupe_du_classement`) ;
  4. `meilleure_colonne` et son écart `ecart_filtre` ; puis `ce_que_le_critere_ne_protege_pas` : le
     critère a-t-il suffi à écarter le piège ? Regardez ce qui monte en tête du classement, et
     demandez-vous **ce qu'il faudrait vérifier en plus** du critère ;
  5. `lecture` : pourquoi la colonne du premier balayage gagnait-elle ? par rapport à quoi ?

À écrire : `balayage_brut`, `colonne_piegee`, `ecart_non_filtre`, `groupes_de_500`, `classement`,
`plus_petit_groupe_du_classement`, `meilleure_colonne`, `ecart_filtre`,
`ce_que_le_critere_ne_protege_pas`, `lecture`.


In [17]:
# QUESTION 2 — Balayage des colonnes sur VOTRE échantillon.
# Pour chaque colonne, on calcule le taux d'absence par groupe puis l'écart max - min.
colonnes_candidates = [c for c in mon_train.columns if c != CIBLE]
resultats_bal = []

for col in colonnes_candidates:
    groupes = mon_train.groupby(col, dropna=False, observed=True)[CIBLE]
    taux = groupes.mean() * 100
    effectifs = groupes.size()
    if len(taux) > 0:
        resultats_bal.append({
            "colonne": col,
            "ecart": float(taux.max() - taux.min()),
            "nombre_groupes": int(len(taux)),
            "groupes_de_500": int((effectifs >= 500).sum())
        })

balayage_brut = (pd.DataFrame(resultats_bal)
                 .set_index("colonne")
                 .sort_values("ecart", ascending=False))
display(balayage_brut)

# Le gagnant sans filtre peut être un identifiant ou une valeur très rare.
colonne_piegee = str(balayage_brut.index[0])
ecart_non_filtre = float(balayage_brut.iloc[0]["ecart"])
groupes_de_500 = int(balayage_brut.iloc[0]["groupes_de_500"])

# Nouveau classement : on ne compare que les groupes comptant au moins 500 lignes.
resultats_filtres = []
for col in colonnes_candidates:
    groupes = mon_train.groupby(col, dropna=False, observed=True)[CIBLE]
    taux = groupes.mean() * 100
    effectifs = groupes.size()
    taux_gardes = taux[effectifs >= 500]
    effectifs_gardes = effectifs[effectifs >= 500]
    if len(taux_gardes) >= 2:
        resultats_filtres.append({
            "colonne": col,
            "ecart": float(taux_gardes.max() - taux_gardes.min()),
            "plus_petit_groupe": int(effectifs_gardes.min()),
            "groupes_gardes": int(len(taux_gardes))
        })

classement = (pd.DataFrame(resultats_filtres)
              .set_index("colonne")
              .sort_values("ecart", ascending=False))
display(classement)

meilleure_colonne = str(classement.index[0])
ecart_filtre = float(classement.iloc[0]["ecart"])
plus_petit_groupe_du_classement = int(classement.iloc[0]["plus_petit_groupe"])

ce_que_le_critere_ne_protege_pas = (
    f"Le seuil de 500 lignes limite les fluctuations des petits groupes, mais ne détecte "
    f"pas une fuite de cible, une variable connue après le rendez-vous, un identifiant "
    f"ou un biais de sélection. La première colonne filtrée est {meilleure_colonne} ; "
    f"son sens métier et sa disponibilité avant le rendez-vous doivent encore être vérifiés."
)
lecture = (
    f"Sans filtre, {colonne_piegee} atteint un écart de {ecart_non_filtre:.2f} points. "
    f"Le gagnant peut provenir de groupes minuscules ou de valeurs presque uniques : "
    f"un taux extrême sur quelques lignes est instable. Après le seuil de 500 lignes, "
    f"la meilleure colonne est {meilleure_colonne}, avec {ecart_filtre:.2f} points d'écart. "
    f"Le critère réduit le bruit des petits groupes, mais ne prouve pas qu'une relation est utile."
)

print("Colonne gagnante sans filtre :", colonne_piegee)
print(f"Écart sans filtre : {ecart_non_filtre:.2f} points")
print("Groupes d'au moins 500 lignes pour le gagnant :", groupes_de_500)
print("Meilleure colonne après filtre :", meilleure_colonne)
print(f"Écart filtré : {ecart_filtre:.2f} points")
print("Taille du plus petit groupe retenu :", plus_petit_groupe_du_classement)
print(ce_que_le_critere_ne_protege_pas)
print(lecture)


,ecart,nombre_groupes,groupes_de_500
colonne,,,
dossier_id,100.000000,89936,0
ligne_id,100.000000,89936,0
patient_id,100.000000,68537,0
motif_absence,100.000000,5,5
date_prise,100.000000,10066,0
taille_cm,100.000000,656,0
indice_meteo_jour,100.000000,146,60
cout_acte,100.000000,1401,3
indice_risque_externe,100.000000,8138,0


,ecart,plus_petit_groupe,groupes_gardes
colonne,,,
motif_absence,100.000000,2969,5
duree_consultation,99.379600,518,30
canal_prise,42.277204,11509,4
taux_absence_passe,28.968871,966,3
delai_jours,25.142369,4750,7
constantes_relevees,24.301083,10338,3
clinique,23.067410,872,80
quartier,20.901903,555,21
rappel_envoye,14.832307,30523,2


Colonne gagnante sans filtre : dossier_id
Écart sans filtre : 100.00 points
Groupes d'au moins 500 lignes pour le gagnant : 0
Meilleure colonne après filtre : motif_absence
Écart filtré : 100.00 points
Taille du plus petit groupe retenu : 2969
Le seuil de 500 lignes limite les fluctuations des petits groupes, mais ne détecte pas une fuite de cible, une variable connue après le rendez-vous, un identifiant ou un biais de sélection. La première colonne filtrée est motif_absence ; son sens métier et sa disponibilité avant le rendez-vous doivent encore être vérifiés.
Sans filtre, dossier_id atteint un écart de 100.00 points. Le gagnant peut provenir de groupes minuscules ou de valeurs presque uniques : un taux extrême sur quelques lignes est instable. Après le seuil de 500 lignes, la meilleure colonne est motif_absence, avec 100.00 points d'écart. Le critère réduit le bruit des petits groupes, mais ne prouve pas qu'une relation est utile.


### Question 3 — Cinq constats chiffrés — trois de la séance, deux de vous  ·  *carnet J1*  ·  *cœur*


Cinq constats, **dont au moins deux mesurés par vous**. Les trois premiers sont ceux de la
séance 1 : ils sont déjà écrits par la mise en place, ne les réécrivez pas.

À faire : ajoutez **deux** constats avec `constat(...)` — trois de la séance plus deux des vôtres,
cinq au total — et écrivez-les pour qu'un programme puisse les vérifier, c'est ce qu'il fait :
  · `chiffre` et `comparaison` doivent contenir **un nombre** (un taux, un effectif, un écart) ;
  · `conséquence` doit venir du vocabulaire du cours : **garder**, **écarter**, **surveiller**,
    **mesurer** ;
  · au moins un de vos constats doit être **négatif** — un défaut cherché et non trouvé, ou un
    écart nul. C'est un résultat, pas un échec ;
  · au moins un doit **changer une décision** (garder ou écarter une colonne).

À écrire : les constats, via `constat(...)`.


In [18]:
# QUESTION 3 — Ajouter deux constats personnels aux trois constats du cours.
# Un constat négatif est utile : il indique aussi ce que les données ne montrent pas.
nb_id_dupliques = int(train[ID].duplicated().sum())
constat(
    "aucun identifiant ligne_id dupliqué" if nb_id_dupliques == 0
    else "des identifiants ligne_id sont dupliqués",
    f"{nb_id_dupliques} identifiant(s) dupliqué(s)",
    f"sur {len(train)} lignes d'entraînement",
    "garder ligne_id pour relier les prédictions, mais l'écarter des variables prédictives"
)

# Le taux de valeurs manquantes motive la création d'un indicateur dans la question 10.
nb_solde_manquant = int(train["solde_credit"].isna().sum())
taux_solde_manquant = 100 * train["solde_credit"].isna().mean()
constat(
    "des valeurs manquent dans solde_credit",
    f"{nb_solde_manquant} valeurs manquantes ({taux_solde_manquant:.2f} %)",
    f"sur {len(train)} lignes, soit {taux_solde_manquant:.2f} % du jeu d'entraînement",
    "garder solde_credit et mesurer l'effet d'un indicateur de valeur manquante"
)


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucun identifiant ligne_id dupliqué,0 identifiant(s) dupliqué(s),sur 149894 lignes d'entraînement,"garder ligne_id pour relier les prédictions, m..."


,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucun identifiant ligne_id dupliqué,0 identifiant(s) dupliqué(s),sur 149894 lignes d'entraînement,"garder ligne_id pour relier les prédictions, m..."
4,des valeurs manquent dans solde_credit,8241 valeurs manquantes (5.50 %),"sur 149894 lignes, soit 5.50 % du jeu d'entraî...",garder solde_credit et mesurer l'effet d'un in...


### Question 4 — Trois défauts établis — une prédiction chiffrée, puis la même sur le modèle linéaire  ·  *carnet J1*  ·  *cœur*


Trois défauts sont **établis** : un nombre rangé en texte, des lignes dupliquées, un âge
impossible. Un diagnostic **certain** ne dit rien de ce qu'il **coûte**.

À faire :
  1. **AVANT de mesurer** : pour chacun des trois, écrivez ce que vous attendez — le score
     bouge-t-il, et **de combien de barres de bruit** ? (`pred_age`, `pred_doublons`,
     `pred_taille`, en barres) ;
  2. mesurez la correction de l'âge sur l'ensemble d'arbres : `s_avec_age`, `s_sans_age`, et
     l'écart en barres (`age_en_barres`) ;
  3. **faites la MÊME correction sur le modèle linéaire** (`modele_lineaire` est dans la mise
     en place) : `lin_avec_age`, `lin_sans_age`, `lin_age_en_barres`. Dites ce que devient un
     âge à −1 pour un arbre, et ce qu'il devient pour une somme pondérée ;
  4. `verdict_age` : au vu de **vos deux mesures**, ce défaut compte-t-il — et pour qui ?
     Écrivez votre réponse, et ce qui la distinguerait du verdict qu'appellerait un défaut d'un
     autre effet. Puis `verdict_doublons` : que vaut un contrôle qui ne trouve **rien** ?

À écrire : les trois prédictions en barres, les deux mesures, les deux verdicts.


In [19]:
# QUESTION 4 — Prédictions écrites AVANT les mesures.
# On prévoit des effets modestes : un défaut visible n'améliore pas forcément le score.
pred_age = 0.5
pred_doublons = 0.2
pred_taille = 0.2

# Pour garder une comparaison propre même si une feature a été ajoutée plus loin dans le carnet,
# on repart de la table sans feature maison, avec exactement les mêmes indices train/validation.
X_base_q4 = X_sans_features.copy()
Xtr_q4 = X_base_q4.loc[Xtr.index].copy()
Xval_q4 = X_base_q4.loc[Xval.index].copy()

# Mesure des arbres avec la valeur d'âge d'origine, puis après remplacement des âges négatifs
# par la médiane des âges valides observés dans le sous-ensemble d'entraînement.
arbre_avec_age = HistGradientBoostingClassifier(random_state=0).fit(Xtr_q4, ytr)
s_avec_age = 100 * arbre_avec_age.score(Xval_q4, yval)

ages_valides = Xtr_q4.loc[Xtr_q4["age"] >= 0, "age"]
age_median = float(ages_valides.median())
Xtr_age_corr = Xtr_q4.copy()
Xval_age_corr = Xval_q4.copy()
Xtr_age_corr.loc[Xtr_age_corr["age"] < 0, "age"] = age_median
Xval_age_corr.loc[Xval_age_corr["age"] < 0, "age"] = age_median
arbre_age_corr = HistGradientBoostingClassifier(random_state=0).fit(Xtr_age_corr, ytr)
s_sans_age = 100 * arbre_age_corr.score(Xval_age_corr, yval)
age_en_barres = (s_sans_age - s_avec_age) / erreur_type

# Même expérience pour la régression logistique, en gardant le même découpage.
Xnum_tr_q4 = X_num.loc[Xtr.index].copy()
Xnum_val_q4 = X_num.loc[Xval.index].copy()
lin_avec_modele = LogisticRegression(solver="newton-cholesky", max_iter=500).fit(Xnum_tr_q4, ytr)
lin_avec_age = 100 * lin_avec_modele.score(Xnum_val_q4, yval)

Xnum_tr_corr = Xnum_tr_q4.copy()
Xnum_val_corr = Xnum_val_q4.copy()
age_median_lin = float(Xnum_tr_corr.loc[Xnum_tr_corr["age"] >= 0, "age"].median())
Xnum_tr_corr.loc[Xnum_tr_corr["age"] < 0, "age"] = age_median_lin
Xnum_val_corr.loc[Xnum_val_corr["age"] < 0, "age"] = age_median_lin
lin_sans_modele = LogisticRegression(solver="newton-cholesky", max_iter=500).fit(Xnum_tr_corr, ytr)
lin_sans_age = 100 * lin_sans_modele.score(Xnum_val_corr, yval)
lin_age_en_barres = (lin_sans_age - lin_avec_age) / erreur_type

# La valeur -1 est un code impossible, pas un âge réel : les arbres peuvent la traiter comme
# une catégorie/seuil particulier, tandis que le modèle linéaire l'interprète comme une valeur
# numérique très basse dans sa somme pondérée.
if abs(age_en_barres) < 2 and abs(lin_age_en_barres) < 2:
    verdict_age = (
        f"Après correction de l'âge négatif, le score des arbres passe de {s_avec_age:.2f} % "
        f"à {s_sans_age:.2f} % ({age_en_barres:.2f} barre), et le score linéaire de "
        f"{lin_avec_age:.2f} % à {lin_sans_age:.2f} % ({lin_age_en_barres:.2f} barre). "
        f"Les deux écarts restent sous deux barres : le défaut existe, mais son coût mesuré "
        f"sur cette validation n'est pas net."
    )
else:
    verdict_age = (
        f"La correction fait passer les arbres de {s_avec_age:.2f} % à {s_sans_age:.2f} % "
        f"({age_en_barres:.2f} barre) et le linéaire de {lin_avec_age:.2f} % à "
        f"{lin_sans_age:.2f} % ({lin_age_en_barres:.2f} barre). Au moins une mesure dépasse "
        f"deux barres : le défaut compte pour ce modèle, mais pas nécessairement de la même façon "
        f"pour les arbres et pour une somme pondérée."
    )

nb_doublons_exacts = int(train.duplicated().sum())
verdict_doublons = (
    f"Le contrôle trouve {nb_doublons_exacts} ligne(s) entièrement dupliquée(s). "
    f"Un résultat nul serait aussi informatif : il indiquerait qu'aucun doublon exact n'a été "
    f"détecté par ce contrôle, sans exclure des doublons sur certaines colonnes ou des dossiers "
    f"répétés sous des identifiants différents."
)

print(f"Prédictions annoncées (en barres) : âge={pred_age}, doublons={pred_doublons}, taille={pred_taille}")
print(f"Arbres — âge d'origine : {s_avec_age:.2f} % ; âge corrigé : {s_sans_age:.2f} % ; écart : {age_en_barres:.2f} barre(s)")
print(f"Linéaire — âge d'origine : {lin_avec_age:.2f} % ; âge corrigé : {lin_sans_age:.2f} % ; écart : {lin_age_en_barres:.2f} barre(s)")
print(verdict_age)
print(verdict_doublons)

# On remet le modèle de base pour les questions 5 à 9, qui comparent le socle du cours.
modele = arbre_avec_age


Prédictions annoncées (en barres) : âge=0.5, doublons=0.2, taille=0.2
Arbres — âge d'origine : 82.51 % ; âge corrigé : 82.49 % ; écart : -0.06 barre(s)
Linéaire — âge d'origine : 82.21 % ; âge corrigé : 82.21 % ; écart : 0.00 barre(s)
Après correction de l'âge négatif, le score des arbres passe de 82.51 % à 82.49 % (-0.06 barre), et le score linéaire de 82.21 % à 82.21 % (0.00 barre). Les deux écarts restent sous deux barres : le défaut existe, mais son coût mesuré sur cette validation n'est pas net.
Le contrôle trouve 0 ligne(s) entièrement dupliquée(s). Un résultat nul serait aussi informatif : il indiquerait qu'aucun doublon exact n'a été détecté par ce contrôle, sans exclure des doublons sur certaines colonnes ou des dossiers répétés sous des identifiants différents.


### Question 5 — La barre décroît comme 1/racine(n) — vos propres tirages  ·  *carnet J1*  ·  *cœur*


L'entraînement ne change pas. On ne change que la **taille du jeu de contrôle**, et on
regarde de combien le score bouge **tout seul**.

À faire :
  1. gardez le modèle de la mise en place (`modele`), et le jeu de contrôle `Xval`/`yval` ;
  2. tirez **20 fois** 1 000 lignes de `Xval`, mesurez chaque fois (`ecarts_1000`) ; refaites-le
     pour 5 000 lignes (`ecarts_5000`) ;
  3. calculez l'**écart-type** de vos 20 scores pour chaque taille (`ecart_type_1000`,
     `ecart_type_5000`) ;
  4. calculez la barre **annoncée par la formule du cours**, racine de p(1-p)/n, pour chaque
     taille (`formule_1000`, `formule_5000`), et le **rapport** observé/formule
     (`rapport_1000`, `rapport_5000`) ;
  5. `lecture` : dans quel sens la barre bouge quand `n` est multiplié par 5 ? Le rapport
     vaut-il à peu près 1 ? Et **vos 20 tirages suffisent-ils** pour estimer un écart-type ?

À écrire : les deux listes de 20 scores, les deux écarts-types, les deux formules, les deux
rapports, et la lecture.


In [20]:
# QUESTION 5 — Vingt sous-échantillons de 1 000 puis 5 000 lignes.
# Le modèle reste fixe : seule la taille du jeu de contrôle change.
rng_q5 = np.random.default_rng(2026)

def scores_sur_tirages(n, repetitions=20):
    scores = []
    for _ in range(repetitions):
        positions = rng_q5.choice(len(Xval), size=n, replace=False)
        Xpetit = Xval.iloc[positions]
        ypetit = yval.iloc[positions]
        scores.append(100 * np.mean(modele.predict(Xpetit) == ypetit.to_numpy()))
    return scores

ecarts_1000 = scores_sur_tirages(1000)
ecarts_5000 = scores_sur_tirages(5000)
ecart_type_1000 = float(np.std(ecarts_1000, ddof=1))
ecart_type_5000 = float(np.std(ecarts_5000, ddof=1))

# Formule binomiale théorique, convertie en points de pourcentage.
p = score_interne / 100
formule_1000 = 100 * np.sqrt(p * (1 - p) / 1000)
formule_5000 = 100 * np.sqrt(p * (1 - p) / 5000)
rapport_1000 = ecart_type_1000 / formule_1000
rapport_5000 = ecart_type_5000 / formule_5000

lecture = (
    f"Quand la taille est multipliée par 5, la barre théorique est divisée par √5 "
    f"(de {formule_1000:.3f} à {formule_5000:.3f} point). Les rapports observé/théorique "
    f"sont {rapport_1000:.2f} et {rapport_5000:.2f} : ils se comparent à 1, mais 20 tirages "
    f"seulement donnent une estimation encore variable de l'écart-type. Les tirages sont "
    f"effectués sans remise dans le jeu de validation."
)
print("Scores sur 1 000 lignes :", [round(x, 3) for x in ecarts_1000])
print("Scores sur 5 000 lignes :", [round(x, 3) for x in ecarts_5000])
print(f"Écart-type observé (1 000) : {ecart_type_1000:.3f} point ; formule : {formule_1000:.3f} ; rapport : {rapport_1000:.2f}")
print(f"Écart-type observé (5 000) : {ecart_type_5000:.3f} point ; formule : {formule_5000:.3f} ; rapport : {rapport_5000:.2f}")
print(lecture)


Scores sur 1 000 lignes : [np.float64(82.6), np.float64(82.0), np.float64(82.4), np.float64(82.7), np.float64(82.1), np.float64(83.8), np.float64(82.2), np.float64(84.0), np.float64(82.0), np.float64(84.3), np.float64(83.0), np.float64(85.1), np.float64(80.8), np.float64(82.0), np.float64(83.4), np.float64(81.4), np.float64(81.6), np.float64(81.2), np.float64(82.8), np.float64(83.9)]
Scores sur 5 000 lignes : [np.float64(82.84), np.float64(82.4), np.float64(82.62), np.float64(82.8), np.float64(82.88), np.float64(82.82), np.float64(82.96), np.float64(82.78), np.float64(82.16), np.float64(82.42), np.float64(83.1), np.float64(81.9), np.float64(81.9), np.float64(82.34), np.float64(83.14), np.float64(82.9), np.float64(82.38), np.float64(82.28), np.float64(82.72), np.float64(82.78)]
Écart-type observé (1 000) : 1.128 point ; formule : 1.201 ; rapport : 0.94
Écart-type observé (5 000) : 0.365 point ; formule : 0.537 ; rapport : 0.68
Quand la taille est multipliée par 5, la barre théorique est

### Question 6 — La formule qui déborde : ce qu'on ne peut plus tirer  ·  *carnet J1*  ·  *approfondissement*


Reprenez le même tirage, mais avec **25 000 lignes** — presque tout le jeu de contrôle.

À faire :
  1. tirez 20 fois 25 000 lignes, calculez l'écart-type (`ecart_type_25000`) ;
  2. calculez la barre de la formule (`formule_25000`) et le rapport
     (`rapport_25000`) ;
  3. `pourquoi_la_formule_deborde` : en une phrase, ce que la formule suppose et que ce tirage-là
     ne fait plus. Comparez les trois rapports que vous venez de calculer (1 000, 5 000, 25 000) :
     que vaut chacun, et qu'est-ce qui a changé entre le deuxième et le troisième ?
  4. **bonus si vous le voulez** : la **correction de population finie** — l'écart-type d'un tirage
     **sans remise** dans un jeu de `N` lignes se multiplie par racine de (N-n)/(N-1). Calculez-la
     pour n = 25 000, et regardez le rapport qu'elle donne.

À écrire : les 20 scores, l'écart-type, la formule, le rapport, et l'explication.


In [21]:
# QUESTION 6 — Tirages de 25 000 lignes, proches de la taille du jeu de validation.
# Un tirage sans remise dans un jeu fini varie moins que ne le prévoit la formule indépendante.
ecarts_25000 = scores_sur_tirages(25000)
ecart_type_25000 = float(np.std(ecarts_25000, ddof=1))
formule_25000 = 100 * np.sqrt(p * (1 - p) / 25000)
rapport_25000 = ecart_type_25000 / formule_25000

N_validation = len(Xval)
correction_population_finie = np.sqrt((N_validation - 25000) / (N_validation - 1))
formule_corrigee_25000 = formule_25000 * correction_population_finie
rapport_corrige_25000 = ecart_type_25000 / formule_corrigee_25000

pourquoi_la_formule_deborde = (
    f"La formule binomiale suppose des observations indépendantes, alors que les 25 000 lignes "
    f"sont tirées sans remise dans seulement {N_validation} lignes : chaque tirage couvre une "
    f"grande part du jeu et la correction de population finie devient importante. Les rapports "
    f"observé/théorique sont {rapport_1000:.2f} (1 000), {rapport_5000:.2f} (5 000) et "
    f"{rapport_25000:.2f} (25 000). La correction finie vaut {correction_population_finie:.3f} "
    f"et donne un rapport corrigé de {rapport_corrige_25000:.2f}."
)
print("Scores sur 25 000 lignes :", [round(x, 3) for x in ecarts_25000])
print(f"Écart-type observé : {ecart_type_25000:.3f} point")
print(f"Formule sans correction : {formule_25000:.3f} point ; rapport : {rapport_25000:.2f}")
print(f"Correction de population finie : {correction_population_finie:.3f}")
print(f"Formule corrigée : {formule_corrigee_25000:.3f} point ; rapport corrigé : {rapport_corrige_25000:.2f}")
print(pourquoi_la_formule_deborde)


Scores sur 25 000 lignes : [np.float64(82.588), np.float64(82.352), np.float64(82.636), np.float64(82.476), np.float64(82.384), np.float64(82.524), np.float64(82.648), np.float64(82.6), np.float64(82.5), np.float64(82.576), np.float64(82.512), np.float64(82.416), np.float64(82.48), np.float64(82.36), np.float64(82.46), np.float64(82.512), np.float64(82.524), np.float64(82.54), np.float64(82.396), np.float64(82.348)]
Écart-type observé : 0.093 point
Formule sans correction : 0.240 point ; rapport : 0.39
Correction de population finie : 0.408
Formule corrigée : 0.098 point ; rapport corrigé : 0.95
La formule binomiale suppose des observations indépendantes, alors que les 25 000 lignes sont tirées sans remise dans seulement 29979 lignes : chaque tirage couvre une grande part du jeu et la correction de population finie devient importante. Les rapports observé/théorique sont 0.94 (1 000), 0.68 (5 000) et 0.39 (25 000). La correction finie vaut 0.408 et donne un rapport corrigé de 0.95.


### Question 7 — Le pas d'apprentissage : qu'est-ce qui change ?  ·  *carnet J1*  ·  *cœur*


Trois valeurs de `learning_rate` — **0,05 / 0,5 / 2** — le reste inchangé. `learning_rate` est le
**pas d'apprentissage** : de combien chaque arbre corrige l'erreur du précédent. C'est le seul
réglage que J1 §6 n'a pas montré (elle n'y a montré que `max_iter`, le nombre d'arbres *demandé*).
Et l'arrêt anticipé s'en mêle : le modèle peut s'arrêter **tout seul** quand sa part interne
cesse de progresser, si bien que le nombre d'arbres *construits* peut différer de celui qu'on
a demandé.

À faire :
  0. **AVANT de mesurer** : `pred_pas` — ce que vous attendez des trois scores et du nombre
     d'arbres construits, avec la raison ;

Puis, dans UN SEUL tableau (`reglages`), une ligne par valeur :
  1. le score de validation, en pourcentage ;
  2. le nombre d'arbres **effectivement construits** (`m.n_iter_`) — pas celui demandé ;
  3. l'écart à la référence, **divisé par la barre de bruit** : combien de barres ?

Puis deux phrases, et elles sont le cœur de la question :
  · `score_separe` : les trois scores sont-ils séparés de **plus de deux barres** ?
  · `ce_qui_borne` : qu'est-ce qui borne **vraiment** ce modèle — le pas, ou autre chose ?
    Appuyez-vous sur ce que vous avez observé, `n_iter_` compris, et dites si cela confirme
    votre `pred_pas`.

À écrire : `pred_pas`, le tableau, `score_separe`, `ce_qui_borne`.


In [22]:
# QUESTION 7 — Prédiction avant de comparer les pas d'apprentissage.
pred_pas = (
    "Je m'attends à ce que le pas 0,05 apprenne lentement, 0,5 corrige plus vite, "
    "et 2 soit potentiellement trop agressif. Les scores et n_iter_ peuvent toutefois "
    "être proches, car l'arrêt anticipé et la complexité des données limitent l'apprentissage."
)

# On ne change que learning_rate ; les données et la graine restent identiques.
reglages = []
modeles_pas = {}
for pas in [0.05, 0.5, 2.0]:
    m = HistGradientBoostingClassifier(learning_rate=pas, random_state=0)
    m.fit(Xtr, ytr)
    score = 100 * m.score(Xval, yval)
    modeles_pas[pas] = m
    reglages.append({
        "learning_rate": pas,
        "score": score,
        "arbres": int(m.n_iter_),
        "en_barres": (score - reference) / erreur_type
    })
reglages = pd.DataFrame(reglages)

scores_pas = reglages["score"].to_numpy()
paires_pas = []
for i in range(len(reglages)):
    for j in range(i + 1, len(reglages)):
        nom_i = str(reglages.iloc[i]["learning_rate"])
        nom_j = str(reglages.iloc[j]["learning_rate"])
        ecart_paire = abs(float(reglages.iloc[i]["score"]) - float(reglages.iloc[j]["score"]))
        paires_pas.append(
            f"{nom_i} vs {nom_j} : {ecart_paire:.3f} point(s)"
            + (" (> 2 barres)" if ecart_paire > 2 * erreur_type else " (≤ 2 barres)")
        )
score_separe = (
    f"Le seuil de deux barres vaut {2 * erreur_type:.3f} points. "
    + "; ".join(paires_pas)
)
ce_qui_borne = (
    "Le modèle est borné par le pas d'apprentissage, mais aussi par l'arrêt anticipé, le nombre "
    "d'arbres effectivement construits (n_iter_), les données et la validation. Si les n_iter_ "
    "diffèrent des 100 arbres demandés ou si les scores restent proches, cela montre que le pas "
    "ne suffit pas à expliquer seul la performance ; la prédiction initiale doit être confrontée "
    "au tableau mesuré."
)
display(reglages)
print(pred_pas)
print(score_separe)
print(ce_qui_borne)


,learning_rate,score,arbres,en_barres
0,0.05,82.497748,71,3.412045
1,0.50,82.090797,15,1.557313
2,2.00,82.137496,10,1.770151


Je m'attends à ce que le pas 0,05 apprenne lentement, 0,5 corrige plus vite, et 2 soit potentiellement trop agressif. Les scores et n_iter_ peuvent toutefois être proches, car l'arrêt anticipé et la complexité des données limitent l'apprentissage.
Le seuil de deux barres vaut 0.439 points. 0.05 vs 0.5 : 0.407 point(s) (≤ 2 barres); 0.05 vs 2.0 : 0.360 point(s) (≤ 2 barres); 0.5 vs 2.0 : 0.047 point(s) (≤ 2 barres)
Le modèle est borné par le pas d'apprentissage, mais aussi par l'arrêt anticipé, le nombre d'arbres effectivement construits (n_iter_), les données et la validation. Si les n_iter_ diffèrent des 100 arbres demandés ou si les scores restent proches, cela montre que le pas ne suffit pas à expliquer seul la performance ; la prédiction initiale doit être confrontée au tableau mesuré.


### Question 8 — Trois essais de trois natures — et le meilleur de N  ·  *carnet J1*  ·  *cœur*


Au moins **trois essais à vous**, et pas trois fois la même idée. La règle de l'ordre est
**appliquée** : `noter(...)` refuse un essai que vous n'avez pas annoncé.

À faire :
  1. trois essais de **natures différentes** : une **donnée** (une colonne, une correction), un
     **réglage** (un paramètre du modèle), une **famille de modèle** (un autre algorithme) ;
  2. `prevoir(...)` d'abord, puis la mesure dans `essai(...)` — il date la mesure lui-même ;
  3. remplissez `mes_essais` (une **table**, pas une liste) : une ligne par essai, avec **l'essai,
     sa nature, l'attendu, le mesuré, le gain en barres, et le verdict** — « réel » ou « dans le
     bruit ». Le gain se compte **par rapport au modèle de la mise en place** (`score_interne`), pas
     par rapport à la référence : c'est le *changement* qui décide du verdict — et un changement nul
     est une réponse, pas un échec ;
  4. `optimisme` : votre meilleur essai a été choisi **parce qu'il gagne sur cette validation**.
     Écrivez ce que ce choix fait à votre estimation du gain — une phrase, **avant** de mesurer
     (`optimisme`) — puis refaites la mesure du même essai sur un **second découpage** (une autre
     graine) : `verifie_par_second_tirage`, le gain de ce tirage, en barres.

À écrire : `mes_essais`, `optimisme`, `verifie_par_second_tirage`.


In [23]:
# QUESTION 8 — Trois essais de natures différentes.
# Pour chaque essai, on annonce d'abord une prédiction, puis essai(...) mesure le score.
from sklearn.ensemble import RandomForestClassifier

scores_essais = {}

nom_donnee = "retirer age"
prevoir(nom_donnee,
        "Je prévois un score proche du modèle de base, avec une petite hausse ou baisse.",
        "L'âge peut apporter de l'information, mais age_mois en contient déjà une partie.")
scores_essais[nom_donnee] = essai(
    nom_donnee,
    lambda: HistGradientBoostingClassifier(random_state=0)
        .fit(Xtr.drop(columns=["age"]), ytr)
        .score(Xval.drop(columns=["age"]), yval),
    changement="Donnée : retirer la colonne age",
    attendu="Je prévois un score proche, car age_mois décrit déjà une information semblable."
)

nom_reglage = "max_iter 150"
prevoir(nom_reglage,
        "Je prévois un changement modeste, éventuellement positif, avec davantage d'arbres.",
        "Un plus grand nombre d'itérations peut améliorer l'ajustement, mais aussi surapprendre.")
scores_essais[nom_reglage] = essai(
    nom_reglage,
    lambda: HistGradientBoostingClassifier(max_iter=150, random_state=0)
        .fit(Xtr, ytr)
        .score(Xval, yval),
    changement="Réglage : max_iter=150 au lieu de la valeur par défaut",
    attendu="Je prévois une petite amélioration si les arbres supplémentaires généralisent."
)

nom_modele = "RandomForest 100 arbres"
prevoir(nom_modele,
        "Je prévois une performance proche ou légèrement différente du modèle de base.",
        "La forêt aléatoire est une autre famille d'arbres, avec une façon différente de combiner les arbres.")
scores_essais[nom_modele] = essai(
    nom_modele,
    lambda: RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0
    ).fit(X_num.loc[Xtr.index], ytr).score(X_num.loc[Xval.index], yval),
    changement="Famille de modèle : RandomForestClassifier",
    attendu="Je prévois un score proche, sans garantie d'amélioration."
)

# Le gain compare chaque essai au score de base, pas à la référence majoritaire.
natures = {
    nom_donnee: "donnée",
    nom_reglage: "réglage",
    nom_modele: "famille de modèle"
}
lignes_essais = [e for e in ESSAIS if e["hypothèse"] in natures]
mes_essais = pd.DataFrame([{
    "essai": e["hypothèse"],
    "nature": natures[e["hypothèse"]],
    "attendu": e["attendu"],
    "mesuré (%)": e["mesuré"],
    "gain_en_barres": (e["mesuré"] - score_interne) / erreur_type,
    "verdict": "réel" if abs((e["mesuré"] - score_interne) / erreur_type) >= 2 else "dans le bruit"
} for e in lignes_essais])

# Choisir le meilleur sur cette validation surestime souvent le gain : il a été choisi parce qu'il
# a eu un bon résultat ici. On le vérifie donc sur un second découpage.
optimisme = (
    "Choisir le meilleur essai parmi plusieurs sur la même validation crée un optimisme : "
    "une partie de son avantage peut venir du hasard de ce découpage. Je vérifie donc le gagnant "
    "sur une validation issue d'une autre graine."
)
meilleur_essai = max(scores_essais, key=lambda k: scores_essais[k])
Xa2, Xb2, ya2, yb2 = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
base2 = HistGradientBoostingClassifier(random_state=0).fit(Xa2, ya2)
score_base2 = 100 * base2.score(Xb2, yb2)

if meilleur_essai == nom_donnee:
    best2 = HistGradientBoostingClassifier(random_state=0).fit(
        Xa2.drop(columns=["age"]), ya2
    )
    score_best2 = 100 * best2.score(Xb2.drop(columns=["age"]), yb2)
elif meilleur_essai == nom_reglage:
    best2 = HistGradientBoostingClassifier(max_iter=150, random_state=0).fit(Xa2, ya2)
    score_best2 = 100 * best2.score(Xb2, yb2)
else:
    best2 = RandomForestClassifier(
        n_estimators=100, min_samples_leaf=5, n_jobs=-1, random_state=0
    ).fit(X_num.loc[Xa2.index], ya2)
    score_best2 = 100 * best2.score(X_num.loc[Xb2.index], yb2)

barre2 = 100 * np.sqrt((score_base2 / 100) * (1 - score_base2 / 100) / len(yb2))
verifie_par_second_tirage = (score_best2 - score_base2) / barre2

display(mes_essais)
print("Meilleur essai sur le premier découpage :", meilleur_essai)
print(optimisme)
print(f"Gain du gagnant sur le second découpage : {verifie_par_second_tirage:.2f} barre(s)")


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26
2,retirer age,"Je prévois un score proche du modèle de base, ...","L'âge peut apporter de l'information, mais age...",10:54:48


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"
2,retirer age,Donnée : retirer la colonne age,"Je prévois un score proche, car age_mois décri...",82.51,"oui, à 10:54:48"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26
2,retirer age,"Je prévois un score proche du modèle de base, ...","L'âge peut apporter de l'information, mais age...",10:54:48
3,max_iter 150,"Je prévois un changement modeste, éventuelleme...",Un plus grand nombre d'itérations peut amélior...,10:54:50


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"
2,retirer age,Donnée : retirer la colonne age,"Je prévois un score proche, car age_mois décri...",82.51,"oui, à 10:54:48"
3,max_iter 150,Réglage : max_iter=150 au lieu de la valeur pa...,Je prévois une petite amélioration si les arbr...,82.51,"oui, à 10:54:50"


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26
2,retirer age,"Je prévois un score proche du modèle de base, ...","L'âge peut apporter de l'information, mais age...",10:54:48
3,max_iter 150,"Je prévois un changement modeste, éventuelleme...",Un plus grand nombre d'itérations peut amélior...,10:54:50
4,RandomForest 100 arbres,Je prévois une performance proche ou légèremen...,La forêt aléatoire est une autre famille d'arb...,10:54:51


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"
2,retirer age,Donnée : retirer la colonne age,"Je prévois un score proche, car age_mois décri...",82.51,"oui, à 10:54:48"
3,max_iter 150,Réglage : max_iter=150 au lieu de la valeur pa...,Je prévois une petite amélioration si les arbr...,82.51,"oui, à 10:54:50"
4,RandomForest 100 arbres,Famille de modèle : RandomForestClassifier,"Je prévois un score proche, sans garantie d'am...",82.34,"oui, à 10:54:51"


,essai,nature,attendu,mesuré (%),gain_en_barres,verdict
0,retirer age,donnée,"Je prévois un score proche, car age_mois décri...",82.51,0.010230,dans le bruit
1,max_iter 150,réglage,Je prévois une petite amélioration si les arbr...,82.51,0.010230,dans le bruit
2,RandomForest 100 arbres,famille de modèle,"Je prévois un score proche, sans garantie d'am...",82.34,-0.764566,dans le bruit


Meilleur essai sur le premier découpage : retirer age
Choisir le meilleur essai parmi plusieurs sur la même validation crée un optimisme : une partie de son avantage peut venir du hasard de ce découpage. Je vérifie donc le gagnant sur une validation issue d'une autre graine.
Gain du gagnant sur le second découpage : 0.00 barre(s)


### Question 9 — Ce que le linéaire coûte, en barres — et ce qu'un coefficient ne dit pas  ·  *carnet J2*  ·  *cœur*


**La règle de lecture, posée ici** : un écart inférieur à **deux barres de bruit** n'est pas
une différence. Divisez, et dites combien de barres.

À faire :
  1. l'écart entre les deux familles est mesuré par la mise en place (`score_interne` et
     `score_lineaire`) : `rapport` = cet écart **divisé par la barre**. Est-ce une différence ?
  2. **des coefficients comparables entre eux** : les colonnes n'ont pas les mêmes unités, donc
     la taille brute d'un coefficient ne veut rien dire d'une colonne à l'autre. Deux routes :
     prendre des **indicatrices** (`quartier_…`, `assurance_…` — J2 §1 les construit), ou
     **centrer-réduire** vous-même les colonnes que vous lisez — `(x - x.mean()) / x.std()`, la mise
     à l'échelle que J2 §1 annonce **souhaitable mais non faite**. Nommez les trois colonnes ;
  3. `age` et `age_mois` disent la même chose, `montant_acte` et `cout_acte` aussi : ne les
     utilisez pas pour lire un coefficient, et dites pourquoi (`pourquoi_pas_les_dupliquees`) ;
  4. `quartier_provoque` : le coefficient d'un quartier dit-il que ce quartier **provoque** plus
     d'absences ? **Quelle autre lecture** en est possible ? Une phrase, pas trois ;
  5. `cout_et_rapport` : ce que le linéaire coûte (indicatrices, comblement, mise à
     l'échelle) et ce qu'il rapporte (des coefficients signés, un par colonne).

À écrire : `rapport`, les trois coefficients, `pourquoi_pas_les_dupliquees`, `quartier_provoque`,
`cout_et_rapport`.


In [24]:
# QUESTION 9 — Comparer les familles de modèles et lire des coefficients comparables.
rapport = abs(score_interne - score_lineaire) / erreur_type

# Les indicatrices de quartier sont toutes de même nature (0/1) : leurs coefficients sont
# donc plus comparables que ceux de variables exprimées en unités différentes.
quartier_cols = [c for c in X_num.columns if str(c).startswith("quartier_")]
if len(quartier_cols) >= 3:
    colonnes_coeff = quartier_cols[:3]
else:
    # Repli si le jeu de données ne contient pas trois indicatrices de quartier.
    colonnes_coeff = [c for c in X_num.columns if c not in ("age", "age_mois", "montant_acte", "cout_acte")][:3]
trois_coefficients = {
    str(c): float(modele_lineaire.coef_[0][X_num.columns.get_loc(c)])
    for c in colonnes_coeff
}

pourquoi_pas_les_dupliquees = (
    "age et age_mois représentent le même âge, tandis que montant_acte et cout_acte sont "
    "parfaitement corrélés dans les données. Les garder ensemble rend l'interprétation des "
    "coefficients instable ou redondante : le modèle peut répartir le même signal entre deux "
    "colonnes, sans qu'on puisse attribuer clairement l'effet à l'une."
)
quartier_provoque = (
    "Non : un coefficient associé à un quartier mesure une association conditionnelle dans les "
    "données observées, pas une cause. Il peut refléter la clinique, le délai, l'accès aux soins "
    "ou d'autres différences entre les patients de ce quartier."
)
cout_et_rapport = (
    f"Le linéaire coûte un encodage des catégories, le traitement des valeurs manquantes et une "
    f"mise à l'échelle souhaitable pour comparer les effets. Il rapporte des coefficients signés "
    f"par colonne, plus faciles à lire que les arbres, mais pas une preuve de causalité. "
    f"Le score des arbres est {score_interne:.2f} %, celui du linéaire {score_lineaire:.2f} % ; "
    f"l'écart vaut {rapport:.2f} barre(s), donc il faut le comparer au seuil de 2 barres."
)
print(f"Écart arbres/linéaire : {rapport:.2f} barre(s)")
print("Trois coefficients de quartier :", trois_coefficients)
print(pourquoi_pas_les_dupliquees)
print(quartier_provoque)
print(cout_et_rapport)


Écart arbres/linéaire : 1.37 barre(s)
Trois coefficients de quartier : {'quartier_Q01': -0.34407569196805715, 'quartier_Q02': -0.45057004150190966, 'quartier_Q03': -0.07192517346048555}
age et age_mois représentent le même âge, tandis que montant_acte et cout_acte sont parfaitement corrélés dans les données. Les garder ensemble rend l'interprétation des coefficients instable ou redondante : le modèle peut répartir le même signal entre deux colonnes, sans qu'on puisse attribuer clairement l'effet à l'une.
Non : un coefficient associé à un quartier mesure une association conditionnelle dans les données observées, pas une cause. Il peut refléter la clinique, le délai, l'accès aux soins ou d'autres différences entre les patients de ce quartier.
Le linéaire coûte un encodage des catégories, le traitement des valeurs manquantes et une mise à l'échelle souhaitable pour comparer les effets. Il rapporte des coefficients signés par colonne, plus faciles à lire que les arbres, mais pas une preuve

### Question 10 — Votre feature — différente de celle du carnet, et son contournement  ·  *carnet J2*  ·  *approfondissement*


**Écrivez une feature que le carnet n'a pas écrite.** `delai_long` est l'exemple de la séance 2,
et la recopier n'apprend rien : votre feature doit découler d'**un de vos constats de la
question 3**, et vous devez dire lequel.

À faire :
  1. dans `ajouter_features` — **la seule cellule de la mise en place faite pour être modifiée** —
     écrivez votre feature, et notez `constat_d_origine` (l'observation du constat qui l'a
     motivée, telle qu'elle est écrite dans le tableau) ;
  2. `prevoir(...)`, puis `remesurer()` : la chaîne passe par `preparer()`, donc `X` reçoit votre
     feature **une seule fois**, et `modele` est mis à jour ;
  3. `gain_de_ma_feature`, en barres de bruit — **un gain de zéro est une réponse** : la table est
     sur un plateau, et une feature raisonnable peut n'y rien changer ;
  4. **contournez `preparer()`** : appelez `modele.predict` sur une table préparée à la main en
     oubliant une étape → `score_contourne`, `ecart_du_contournement` (en barres), et
     `quelle_etape_oubliee`.

À écrire : la feature, son constat d'origine, le gain, le contournement et ce qu'il a produit.


In [25]:
# QUESTION 10 — Feature personnelle motivée par le constat sur solde_credit.
# Le constat exact est repris ci-dessous pour que le lien avec la question 3 soit vérifiable.
constat_d_origine = "des valeurs manquent dans solde_credit"

# La feature est ajoutée dans la fonction ajouter_features de la mise en place :
# solde_credit_manquant vaut 1 quand le solde n'a pas été renseigné, sinon 0.
# Le modèle peut ainsi distinguer une valeur manquante d'un solde réellement égal à zéro.
prevoir(
    "solde_credit_manquant",
    "Je prévois un gain nul ou légèrement positif.",
    "Le constat de la question 3 indique des valeurs manquantes ; un indicateur peut porter ce signal."
)
score_avec_feature = remesurer()
gain_de_ma_feature = (score_avec_feature - score_interne) / erreur_type

# Contournement volontaire : le modèle a appris avec l'indicateur, mais on oublie de le calculer
# et on met 0 pour toutes les lignes. C'est une démonstration du risque de préparer différemment.
Xval_contourne = X_sans_features.loc[Xval.index].copy()
Xval_contourne["solde_credit_manquant"] = 0
Xval_contourne = Xval_contourne[X.columns]
score_contourne = 100 * modele.score(Xval_contourne, yval)
ecart_du_contournement = (score_contourne - score_avec_feature) / erreur_type
quelle_etape_oubliee = (
    "J'ai contourné preparer() et oublié l'étape qui calcule solde_credit_manquant à partir des "
    "valeurs manquantes de solde_credit. En mettant artificiellement l'indicateur à 0, le modèle "
    "ne reçoit plus la même information qu'à l'entraînement ; l'écart mesuré illustre ce bug."
)

print(f"Score de référence avant la feature : {score_interne:.2f} %")
print(f"Score après ajout de la feature : {score_avec_feature:.2f} %")
print(f"Gain de la feature : {gain_de_ma_feature:.2f} barre(s)")
print(f"Score en contournant la préparation : {score_contourne:.2f} %")
print(f"Écart dû au contournement : {ecart_du_contournement:.2f} barre(s)")
print(quelle_etape_oubliee)


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26
2,retirer age,"Je prévois un score proche du modèle de base, ...","L'âge peut apporter de l'information, mais age...",10:54:48
3,max_iter 150,"Je prévois un changement modeste, éventuelleme...",Un plus grand nombre d'itérations peut amélior...,10:54:50
4,RandomForest 100 arbres,Je prévois une performance proche ou légèremen...,La forêt aléatoire est une autre famille d'arb...,10:54:51
5,solde_credit_manquant,Je prévois un gain nul ou légèrement positif.,Le constat de la question 3 indique des valeur...,10:55:23


Score de référence avant la feature : 82.51 %
Score après ajout de la feature : 82.51 %
Gain de la feature : 0.00 barre(s)
Score en contournant la préparation : 82.51 %
Écart dû au contournement : 0.00 barre(s)
J'ai contourné preparer() et oublié l'étape qui calcule solde_credit_manquant à partir des valeurs manquantes de solde_credit. En mettant artificiellement l'indicateur à 0, le modèle ne reçoit plus la même information qu'à l'entraînement ; l'écart mesuré illustre ce bug.


### Question 11 — La colonne qui disparaît : la trouver, mesurer son gain, et le voir s'évaporer  ·  *carnet J2*  ·  *cœur*


**Cherchez-la.** J1 l'a rangée dans `A_ECARTER` — parmi dix autres, et sans dire pourquoi.
Ce qui la trahit est un **écart de remplissage** entre `train` et `test`.

Le gain et la simulation se mesurent sur **`mon_train`** — votre échantillon, comme à la
question 2 : un gain recopié d'un voisin ne vaudra pas le vôtre.

À faire :
  1. calculez, **colonne par colonne**, l'écart de remplissage entre `train` et `test`, triez, et
     nommez celle qui se détache (`colonne_trouvee`) avec son écart (`ecart_remplissage`, en
     points). La **deuxième** du tri mérite la même attention : dites pour elle aussi pourquoi elle
     n'est pas la réponse (`pourquoi_pas_la_derivee`) ;
  2. `remplissage_du_test` : quelle **part du test** porte encore une valeur dans cette colonne ?
  3. `gain_interne` : le même modèle, avec et sans cette colonne, sur la **même** validation —
     le gain en points, et en barres (`gain_en_barres`) ;
  4. **la simulation, et c'est le cœur** : entraînez avec la colonne, puis rendez-la manquante
     dans la validation **pour une part de lignes égale à celle du test** (le taux de trous du
     test). `score_simule`, `chute_en_barres` : que reste-t-il du gain ?
  5. `decision_A_ECARTER` : l'écarter était-il la bonne décision ? Était-ce pour la bonne
     raison ? Une phrase pour chacune des deux questions.

À écrire : la colonne, l'écart, la part du test, le gain interne, la simulation, la décision.


In [26]:
# QUESTION 11 — Chercher la colonne dont le taux de remplissage change entre train et test.
# L'écart est exprimé en points de pourcentage ; la cible n'existe pas dans test et est exclue.
colonnes_communes = [c for c in train.columns if c in test.columns and c != CIBLE]
ecarts = pd.DataFrame(index=colonnes_communes)
ecarts["remplissage_train"] = 100 * train[colonnes_communes].notna().mean()
ecarts["remplissage_test"] = 100 * test[colonnes_communes].notna().mean()
ecarts["ecart"] = (ecarts["remplissage_train"] - ecarts["remplissage_test"]).abs()
ecarts = ecarts.sort_values("ecart", ascending=False)
display(ecarts.head(15))

colonne_trouvee = str(ecarts.index[0])
ecart_remplissage = float(ecarts.iloc[0]["ecart"])
remplissage_du_test = float(100 * test[colonne_trouvee].notna().mean())

# La deuxième colonne est signalée elle aussi, mais il faut distinguer disparition en production
# et champ renseigné après l'issue du rendez-vous.
deuxieme_colonne = str(ecarts.index[1])
ecart_deuxieme = float(ecarts.iloc[1]["ecart"])
if deuxieme_colonne == "motif_absence":
    pourquoi_pas_la_derivee = (
        f"La deuxième colonne, {deuxieme_colonne}, a un écart de {ecart_deuxieme:.2f} points, "
        f"mais elle décrit la raison de l'absence et n'est renseignée qu'après l'issue : c'est "
        f"une fuite potentielle de cible, pas simplement un champ qui disparaît en production."
    )
else:
    pourquoi_pas_la_derivee = (
        f"La deuxième colonne, {deuxieme_colonne}, a un écart de {ecart_deuxieme:.2f} points. "
        f"Elle ne remplace pas {colonne_trouvee}, car son écart est inférieur à celui de la "
        f"première colonne ({ecart_remplissage:.2f} points). Il faut aussi vérifier son sens "
        f"métier et si elle est disponible avant le rendez-vous."
    )

# Pour mesurer le gain, on compare le même modèle, le même échantillon et le même découpage,
# avec puis sans la colonne détectée. La colonne est convertie en catégorie pour ce test.
Xq_base = matrice(mon_train)[0]
yq = mon_train[CIBLE]
idx_q_tr, idx_q_va = train_test_split(
    mon_train.index, test_size=0.2, stratify=yq, random_state=17
)
Xq_sans = Xq_base.drop(columns=[colonne_trouvee], errors="ignore").copy()
Xq_avec = Xq_sans.copy()
serie_col = mon_train[colonne_trouvee].astype("string").fillna("__MANQUANT__").astype(str).astype("category")
serie_col = serie_col.cat.add_categories(["__ABSENT_TEST__"])
Xq_avec[colonne_trouvee] = serie_col

m_sans = HistGradientBoostingClassifier(random_state=0).fit(Xq_sans.loc[idx_q_tr], yq.loc[idx_q_tr])
m_avec = HistGradientBoostingClassifier(random_state=0).fit(Xq_avec.loc[idx_q_tr], yq.loc[idx_q_tr])
score_q_sans = 100 * m_sans.score(Xq_sans.loc[idx_q_va], yq.loc[idx_q_va])
score_q_avec = 100 * m_avec.score(Xq_avec.loc[idx_q_va], yq.loc[idx_q_va])
gain_interne = score_q_avec - score_q_sans
gain_en_barres = gain_interne / erreur_type

# Simulation : on rend manquante dans la validation une part de lignes égale au taux de trous de test.
Xq_simule = Xq_avec.loc[idx_q_va].copy()
taux_trous_test = 1 - (remplissage_du_test / 100)
nb_a_masquer = int(round(taux_trous_test * len(Xq_simule)))
rng_sim = np.random.default_rng(2026)
if nb_a_masquer > 0:
    positions_masquees = rng_sim.choice(Xq_simule.index.to_numpy(), size=nb_a_masquer, replace=False)
    Xq_simule.loc[positions_masquees, colonne_trouvee] = np.nan
score_simule = 100 * m_avec.score(Xq_simule, yq.loc[idx_q_va])
chute_en_barres = (score_q_avec - score_simule) / erreur_type

if remplissage_du_test < 10 and ecarts.loc[colonne_trouvee, "remplissage_train"] > 90:
    decision_A_ECARTER = (
        f"Oui, l'écarter est une bonne décision pour la production : seulement {remplissage_du_test:.2f} % "
        f"des lignes test ont cette colonne renseignée. La raison est sa disponibilité instable, "
        f"pas nécessairement son manque de pouvoir prédictif : son gain interne est {gain_interne:.2f} "
        f"point(s) ({gain_en_barres:.2f} barre(s)) et le score simulé avec les trous est "
        f"{score_simule:.2f} % (chute de {chute_en_barres:.2f} barre(s))."
    )
else:
    decision_A_ECARTER = (
        f"La décision doit se fonder sur la disponibilité réelle : {remplissage_du_test:.2f} % "
        f"des lignes test portent encore cette colonne. Son gain interne vaut {gain_interne:.2f} "
        f"point(s) ({gain_en_barres:.2f} barre(s)) et le score simulé avec des trous vaut "
        f"{score_simule:.2f} % (chute de {chute_en_barres:.2f} barre(s)). L'écarter peut être "
        f"justifié si cette information n'est pas garantie au moment de prédire ; il faut distinguer "
        f"la bonne décision de la raison initialement donnée."
    )

print("Colonne la plus touchée :", colonne_trouvee)
print(f"Écart de remplissage : {ecart_remplissage:.2f} points")
print(f"Part renseignée dans test : {remplissage_du_test:.2f} %")
print(pourquoi_pas_la_derivee)
print(f"Gain interne : {gain_interne:.3f} point(s), soit {gain_en_barres:.2f} barre(s)")
print(f"Score simulé avec les trous du test : {score_simule:.2f} % ; chute : {chute_en_barres:.2f} barre(s)")
print(decision_A_ECARTER)


,remplissage_train,remplissage_test,ecart
canal_prise,100.000000,0.497481,99.502519
motif_absence,18.250897,19.003154,0.752257
rappel_envoye,33.734506,34.067262,0.332756
constantes_relevees,57.616716,57.346872,0.269844
solde_credit,94.502115,94.524568,0.022453
date_naissance,99.969312,99.965474,0.003837
age,100.000000,100.000000,0.000000
district,100.000000,100.000000,0.000000
clinique,100.000000,100.000000,0.000000
patient_id,100.000000,100.000000,0.000000


Colonne la plus touchée : canal_prise
Écart de remplissage : 99.50 points
Part renseignée dans test : 0.50 %
La deuxième colonne, motif_absence, a un écart de 0.75 points, mais elle décrit la raison de l'absence et n'est renseignée qu'après l'issue : c'est une fuite potentielle de cible, pas simplement un champ qui disparaît en production.
Gain interne : 2.991 point(s), soit 13.63 barre(s)
Score simulé avec les trous du test : 82.66 % ; chute : 13.73 barre(s)
Oui, l'écarter est une bonne décision pour la production : seulement 0.50 % des lignes test ont cette colonne renseignée. La raison est sa disponibilité instable, pas nécessairement son manque de pouvoir prédictif : son gain interne est 2.99 point(s) (13.63 barre(s)) et le score simulé avec les trous est 82.66 % (chute de 13.73 barre(s)).


### Question 12 — Votre outil : `suspects(df, cible)`  ·  *carnet J2*  ·  *cœur*


Le carnet a montré **deux signes** d'une colonne qui connaît la réponse, mais il l'a fait une
colonne à la fois. Écrivez l'**outil** qui les cherche toutes — vous le réutiliserez en projet.

À faire :
  1. `suspects(df, cible)` rend une **table**, une ligne par colonne suspecte, avec au moins :
     le nom, son **écart de remplissage** entre les deux groupes de la cible, et le **signe**
     qui l'a trahie ;
     — **signe 1, le VIDE** : une colonne remplie d'un côté et vide de l'autre (écart de
       remplissage de plus de 50 points) ;
     — **signe 2, la VALEUR** : une colonne remplie **des deux côtés** dont une valeur ne vit
       que dans **un** groupe ;
     — votre outil doit **ignorer l'identifiant** et la cible elle-même ;
  2. appliquez-le à `train` : `table_suspects` ;
  3. Rendez `classement_suspects` — une table d'**une ligne par colonne signalée**, avec deux
     colonnes : `verdict` (à choisir dans **fuite** / **pas une fuite** / **à discuter**) et
     `justification` — et justifiez par **ce que la valeur veut dire**, jamais par le fait qu'elle
     est constante ;
  4. **le troisième signe, celui que les deux premiers ne voient pas** : une valeur
     **impossible** (une **sentinelle** : un code mis à la place d'une mesure manquante — le −1
     d'`age` en séance 1 en est une) portée par **peu** de lignes. `sentinelle_trouvee`
     (la colonne et la valeur), `lignes_porteuses` (combien de lignes), `taux_sur_ces_lignes`
     (le taux d'absence sur ces lignes, comparé au taux global), et `verdict_sentinelle` : fuite,
     ou défaut de saisie ?

À écrire : la fonction, `table_suspects`, `classement_suspects`, la sentinelle et son verdict.


In [27]:
# QUESTION 12 — Outil générique de recherche de signes de fuite.
# Signe « vide » : remplissage très différent selon la cible. Signe « valeur » :
# une catégorie répétée n'apparaît que dans un seul groupe de la cible.
def suspects(df, cible):
    lignes = []
    groupes_cible = sorted(df[cible].dropna().unique().tolist())
    if len(groupes_cible) != 2:
        raise ValueError("Cette fonction attend une cible binaire avec deux groupes.")
    y0, y1 = groupes_cible[0], groupes_cible[1]
    identifiants = {ID, "dossier_id", "patient_id", cible}

    for col in df.columns:
        if col in identifiants:
            continue
        s0 = df.loc[df[cible] == y0, col]
        s1 = df.loc[df[cible] == y1, col]
        rempl0 = 100 * s0.notna().mean()
        rempl1 = 100 * s1.notna().mean()
        ecart = abs(float(rempl0 - rempl1))
        signes = []

        if ecart > 50:
            signes.append("vide")

        # On ne cherche le signe « valeur » que sur des colonnes catégorielles ou peu distinctes :
        # les nombres continus presque tous uniques ne sont pas des catégories suspectes.
        est_categorielle = (
            not pd.api.types.is_numeric_dtype(df[col])
            or df[col].nunique(dropna=True) <= 30
        )
        valeur_exclusive = None
        effectif_exclusif = 0
        if est_categorielle:
            valeurs = df[col].dropna().unique()
            for valeur in valeurs:
                n0 = int((s0 == valeur).sum())
                n1 = int((s1 == valeur).sum())
                total = n0 + n1
                if total >= 20 and (n0 == 0 or n1 == 0):
                    signes.append("valeur")
                    valeur_exclusive = str(valeur)
                    effectif_exclusif = total
                    break

        if signes:
            lignes.append({
                "colonne": col,
                "ecart_remplissage": round(ecart, 2),
                "signe": " + ".join(sorted(set(signes))),
                "valeur_exclusive": valeur_exclusive,
                "effectif_valeur_exclusive": effectif_exclusif
            })
    return pd.DataFrame(lignes, columns=[
        "colonne", "ecart_remplissage", "signe",
        "valeur_exclusive", "effectif_valeur_exclusive"
    ])

table_suspects = suspects(train, CIBLE)
display(table_suspects)

# Le verdict dépend du sens métier, pas du seul fait qu'une colonne soit constante.
lignes_classement = []
for _, ligne in table_suspects.iterrows():
    col = ligne["colonne"]
    if col in {"motif_absence", "duree_consultation", "constantes_relevees"}:
        verdict = "fuite"
        justification = (
            f"{col} décrit l'issue ou une information recueillie pendant/après la consultation ; "
            f"elle ne serait pas connue au moment où l'on doit prédire la venue du patient."
        )
    elif col in {"rappel_envoye", "canal_prise", "clinique", "district", "quartier", "assurance", "sexe"}:
        verdict = "à discuter"
        justification = (
            f"{col} peut décrire une situation réelle avant le rendez-vous, mais son association "
            f"à l'absence peut refléter le délai, l'organisation ou un biais ; il faut vérifier "
            f"sa disponibilité et comparer les groupes pertinents."
        )
    else:
        verdict = "pas une fuite"
        justification = (
            f"{col} semble décrire une caractéristique disponible avant l'issue ; le signe statistique "
            f"seul ne prouve pas une fuite, mais la signification et le moment de collecte doivent "
            f"être vérifiés dans le processus métier."
        )
    lignes_classement.append({
        "colonne": col, "verdict": verdict, "justification": justification
    })
classement_suspects = pd.DataFrame(lignes_classement, columns=["colonne", "verdict", "justification"])
display(classement_suspects)

# Troisième signe : une valeur impossible (sentinelle) portée par peu de lignes.
ages_negatifs = train.loc[train["age"] < 0]
if len(ages_negatifs):
    valeur_sentinelle = ages_negatifs["age"].value_counts().index[0]
    lignes_porteuses = int((train["age"] == valeur_sentinelle).sum())
    taux_sur_ces_lignes = 100 * train.loc[train["age"] == valeur_sentinelle, CIBLE].mean()
    sentinelle_trouvee = f"age = {valeur_sentinelle}"
else:
    # Repli si aucune valeur négative n'est présente : rechercher une sentinelle dans age_mois.
    ages_mois_negatifs = train.loc[train["age_mois"] < 0]
    if len(ages_mois_negatifs):
        valeur_sentinelle = ages_mois_negatifs["age_mois"].value_counts().index[0]
        lignes_porteuses = int((train["age_mois"] == valeur_sentinelle).sum())
        taux_sur_ces_lignes = 100 * train.loc[train["age_mois"] == valeur_sentinelle, CIBLE].mean()
        sentinelle_trouvee = f"age_mois = {valeur_sentinelle}"
    else:
        sentinelle_trouvee = "aucune valeur négative trouvée dans age ou age_mois"
        lignes_porteuses = 0
        taux_sur_ces_lignes = float(train[CIBLE].mean() * 100)

taux_global_absence = 100 * train[CIBLE].mean()
verdict_sentinelle = (
    f"Il s'agit vraisemblablement d'un défaut de saisie ou d'une sentinelle, pas d'une fuite : "
    f"une valeur d'âge négative est impossible et sert probablement à coder une mesure inconnue. "
    f"Le taux d'absence sur ces lignes est {taux_sur_ces_lignes:.2f} %, contre {taux_global_absence:.2f} % "
    f"globalement ; cet écart mérite d'être surveillé, mais ne prouve pas à lui seul une fuite."
)
print("Sentinelle :", sentinelle_trouvee)
print("Lignes porteuses :", lignes_porteuses)
print(f"Taux d'absence sur ces lignes : {taux_sur_ces_lignes:.2f} % (global : {taux_global_absence:.2f} %)")
print(verdict_sentinelle)


,colonne,ecart_remplissage,signe,valeur_exclusive,effectif_valeur_exclusive
0,motif_absence,100.0,valeur + vide,non précisé,5510
1,taille_cm,0.0,valeur,"141,1",20
2,date_prise,0.0,valeur,2023-03-16T08:00,22
3,tournee_jour,0.0,valeur,T00853,21


,colonne,verdict,justification
0,motif_absence,fuite,motif_absence décrit l'issue ou une informatio...
1,taille_cm,pas une fuite,taille_cm semble décrire une caractéristique d...
2,date_prise,pas une fuite,date_prise semble décrire une caractéristique ...
3,tournee_jour,pas une fuite,tournee_jour semble décrire une caractéristiqu...


Sentinelle : age = -1
Lignes porteuses : 46
Taux d'absence sur ces lignes : 10.87 % (global : 18.25 %)
Il s'agit vraisemblablement d'un défaut de saisie ou d'une sentinelle, pas d'une fuite : une valeur d'âge négative est impossible et sert probablement à coder une mesure inconnue. Le taux d'absence sur ces lignes est 10.87 %, contre 18.25 % globalement ; cet écart mérite d'être surveillé, mais ne prouve pas à lui seul une fuite.


### Question 13 — Le rappel : cause, ou autre chose ? — et l'expérience qui trancherait  ·  *carnet J2*  ·  *approfondissement*


Un rappel est envoyé, et l'absence est **plus** fréquente avec le rappel. Faut-il en conclure
que le rappel fait manquer les rendez-vous ?

**Tout se mesure sur `mon_train`** — votre échantillon, comme aux questions 2 et 11.

À faire :
  1. **AVANT de regarder les tranches** : `annonce` — le rappel aide-t-il, ou nuit-il, une fois la
     comparaison faite **à délai égal** ? Écrivez votre prédiction, et ce qui vous ferait changer
     d'avis ;
  2. le taux d'absence avec et sans rappel, **globalement** : `ecart_global` (en points) ;
  3. **à délai égal** : par tranches de délai, le taux avec et sans rappel dans **chaque** tranche,
     et pour chaque écart **sa barre de bruit** (`tranches`) — sans la barre, la conclusion
     contredit la règle du cours : un écart de moins de deux barres n'est pas un écart. Comptez
     les **effectifs** : toutes les tranches ne portent pas les deux groupes, la plus courte n'en
     porte qu'un — et sur **votre** échantillon, certaines tomberont **sous les 500 lignes** :
     celles-là ne se comparent pas, mettez-les dans `tranches_exclues` en disant pourquoi ;
  4. `variable_qui_explique` : la variable — s'il y en a une — qui explique l'écart global, avec
     le chiffre qui le montre (`delai_median_avec`, `delai_median_sans`) ; s'il n'y en a pas,
     dites-le ;
  5. `experience_qui_trancherait` : quelle expérience **trancherait vraiment** ? Décrivez-la en
     trois lignes — ce qu'on tirerait au hasard, ce qu'on comparerait, et pourquoi la comparaison
     serait alors honnête.

À écrire : l'annonce, l'écart global, les tranches avec leur barre, la variable, l'expérience.


In [28]:
# QUESTION 13 — Rappel : association globale ou effet à délai égal ?
# Annonce rédigée avant le calcul des tranches : un rappel peut être ciblé sur des rendez-vous
# plus éloignés ; je m'attends donc à un effet global confondu par le délai.
annonce = (
    "Je m'attends à ce que le taux global avec rappel puisse être plus élevé si les rappels sont "
    "surtout envoyés pour des rendez-vous plus éloignés ou à risque. À délai égal, je m'attends "
    "plutôt à un effet plus faible ou protecteur ; je changerais d'avis si plusieurs tranches "
    "suffisamment grandes montraient un écart défavorable supérieur à deux barres."
)

# Ici, une valeur renseignée dans rappel_envoye signifie « rappel envoyé » ; une valeur manquante
# signifie « pas de rappel observé ». Cette convention doit être vérifiée avec le dictionnaire des données.
df_rappel = mon_train[["delai_jours", "rappel_envoye", CIBLE]].copy()
df_rappel["avec_rappel"] = df_rappel["rappel_envoye"].notna()
taux_avec_global = 100 * df_rappel.loc[df_rappel["avec_rappel"], CIBLE].mean()
taux_sans_global = 100 * df_rappel.loc[~df_rappel["avec_rappel"], CIBLE].mean()
ecart_global = float(taux_avec_global - taux_sans_global)

# Tranches fixes de délai, interprétables en jours. On compare les deux groupes dans chaque tranche.
bornes = [-np.inf, 0, 1, 3, 7, 14, 30, np.inf]
labels = ["≤ 0 jour", "1 jour", "2–3 jours", "4–7 jours", "8–14 jours", "15–30 jours", "> 30 jours"]
df_rappel["tranche_delai"] = pd.cut(df_rappel["delai_jours"], bins=bornes, labels=labels)
tranches = []
tranches_exclues = []

for etiquette in labels:
    sous = df_rappel[df_rappel["tranche_delai"] == etiquette]
    avec = sous[sous["avec_rappel"]]
    sans = sous[~sous["avec_rappel"]]
    n_avec, n_sans = len(avec), len(sans)
    total = n_avec + n_sans

    if total < 500 or n_avec == 0 or n_sans == 0:
        raison = (
            f"Comparaison exclue : {total} ligne(s), {n_avec} avec rappel et {n_sans} sans ; "
            f"il faut au moins 500 lignes au total et les deux groupes."
        )
        tranches_exclues.append({"tranche": etiquette, "lignes": total, "raison": raison})
        continue

    taux_avec = 100 * avec[CIBLE].mean()
    taux_sans = 100 * sans[CIBLE].mean()
    ecart = float(taux_avec - taux_sans)
    barre = 100 * np.sqrt(
        (avec[CIBLE].mean() * (1 - avec[CIBLE].mean()) / n_avec)
        + (sans[CIBLE].mean() * (1 - sans[CIBLE].mean()) / n_sans)
    )
    tranches.append({
        "tranche": str(etiquette),
        "taux_avec": float(taux_avec),
        "taux_sans": float(taux_sans),
        "effectif_avec": int(n_avec),
        "effectif_sans": int(n_sans),
        "lignes": int(total),
        "ecart": ecart,
        "barre": float(barre),
        "en_barres": float(ecart / barre) if barre > 0 else 0.0
    })

# Les médianes indiquent si les rendez-vous avec rappel ont des délais différents.
delai_median_avec = float(df_rappel.loc[df_rappel["avec_rappel"], "delai_jours"].median())
delai_median_sans = float(df_rappel.loc[~df_rappel["avec_rappel"], "delai_jours"].median())
variable_qui_explique = (
    f"Le délai est un facteur de confusion plausible : médiane de {delai_median_avec:.1f} jours "
    f"avec rappel contre {delai_median_sans:.1f} jours sans rappel. Si ces médianes diffèrent, "
    f"les deux groupes ne sont pas directement comparables globalement ; les tranches et leurs "
    f"barres permettent de voir si l'écart persiste à délai égal."
)
experience_qui_trancherait = (
    "Tirer au hasard, parmi des rendez-vous comparables et dans chaque tranche de délai, ceux qui "
    "recevront un rappel et ceux qui n'en recevront pas. Comparer ensuite le taux d'absence entre "
    "les deux groupes en conservant les mêmes règles de suivi. La randomisation équilibre en moyenne "
    "les autres facteurs et permet une comparaison plus honnête de l'effet du rappel."
)

display(pd.DataFrame(tranches))
display(pd.DataFrame(tranches_exclues))
print(annonce)
print(f"Taux avec rappel : {taux_avec_global:.2f} % ; sans rappel : {taux_sans_global:.2f} % ; écart : {ecart_global:.2f} points")
print(variable_qui_explique)
print(experience_qui_trancherait)


,tranche,taux_avec,taux_sans,effectif_avec,effectif_sans,lignes,ecart,barre,en_barres
0,4–7 jours,24.803477,28.750000,10813,880,11693,-3.946523,1.581221,-2.495870
1,8–14 jours,28.595580,35.373134,8281,670,8951,-6.777554,1.912745,-3.543366
2,15–30 jours,30.741999,37.580645,7062,620,7682,-6.838646,2.021132,-3.383572
3,> 30 jours,30.112205,34.464752,4367,383,4750,-4.352547,2.525704,-1.723301


,tranche,lignes,raison
0,≤ 0 jour,35747,"Comparaison exclue : 35747 ligne(s), 0 avec ra..."
1,1 jour,10348,"Comparaison exclue : 10348 ligne(s), 0 avec ra..."
2,2–3 jours,10765,"Comparaison exclue : 10765 ligne(s), 0 avec ra..."


Je m'attends à ce que le taux global avec rappel puisse être plus élevé si les rappels sont surtout envoyés pour des rendez-vous plus éloignés ou à risque. À délai égal, je m'attends plutôt à un effet plus faible ou protecteur ; je changerais d'avis si plusieurs tranches suffisamment grandes montraient un écart défavorable supérieur à deux barres.
Taux avec rappel : 27.97 % ; sans rappel : 13.13 % ; écart : 14.83 points
Le délai est un facteur de confusion plausible : médiane de 11.0 jours avec rappel contre 0.0 jours sans rappel. Si ces médianes diffèrent, les deux groupes ne sont pas directement comparables globalement ; les tranches et leurs barres permettent de voir si l'écart persiste à délai égal.
Tirer au hasard, parmi des rendez-vous comparables et dans chaque tranche de délai, ceux qui recevront un rappel et ceux qui n'en recevront pas. Comparer ensuite le taux d'absence entre les deux groupes en conservant les mêmes règles de suivi. La randomisation équilibre en moyenne les a

### Question 14 — Le rapport final — et une hypothèse testée sous son propre nom  ·  *carnet J1 + J2*  ·  *cœur*


Les trois tableaux se remplissent **dans les questions** ; cette question-ci produit les
hypothèses, et **en teste une**.

À faire :
  1. écrivez **trois** hypothèses de features avec `hypothese(...)` : ce qu'elle raconte (avec le
     **signe** attendu), comment on la teste, et ce qui la **contredirait** — un résultat précis,
     pas « si ça ne marche pas » ;
  2. **testez-en une** avec `essai(...)`, sous **le même nom** que l'hypothèse : la mesure, puis
     `confirmee_ou_contredite` — « confirmée » ou « contredite », jugée sur le critère que
     l'hypothèse avait annoncé **avant** la mesure ;
  3. un « contredite » n'est pas un échec : c'est le seul des deux verdicts qui apprend quelque
     chose. Écrivez ce que vous en faites (`suite_donnee`).

À écrire : les trois hypothèses, le test, le verdict, la suite donnée.


In [29]:
# QUESTION 14 — Trois hypothèses explicites, puis test d'une hypothèse sous le même nom.
hypothese(
    "solde_credit_manquant",
    "Un solde de crédit manquant peut signaler un profil de dossier différent ; j'attends un gain de score d'au moins 0,2 point si cet indicateur apporte une information utile.",
    "Comparer le score de validation avec la feature solde_credit_manquant à celui du modèle de base, sur le même découpage.",
    "L'hypothèse sera contredite si le gain est inférieur à 0,2 point ; un gain nul ou négatif ne confirme pas l'utilité annoncée."
)
hypothese(
    "rappel_a_delai_egal",
    "À délai égal, un rappel pourrait être associé à un taux d'absence plus faible ; le signe attendu est donc un écart avec rappel moins sans rappel inférieur à zéro.",
    "Comparer les taux d'absence avec et sans rappel dans les tranches de délai comportant au moins 500 lignes et les deux groupes.",
    "L'hypothèse sera contredite si les tranches assez grandes montrent toutes un écart nul ou positif, ou si aucun écart ne dépasse deux barres."
)
hypothese(
    "age_superieur_60_ans",
    "Une tranche d'âge peut porter un signal d'organisation ou d'accès aux soins ; je prévois une différence de taux d'absence pour les patients de plus de 60 ans.",
    "Créer un indicateur age_superieur_60, comparer le score avec et sans cet indicateur sur la même validation.",
    "L'hypothèse sera contredite si l'ajout de l'indicateur n'améliore pas le score d'au moins 0,2 point et si l'écart de taux est inférieur à une barre."
)

# Test sous le même nom que l'hypothèse : la feature a été ajoutée à X par la question 10.
prevoir(
    "solde_credit_manquant",
    "Je prévois un gain d'au moins 0,2 point, comme annoncé dans l'hypothèse.",
    "Vérifier sur la validation fixe si l'indicateur de solde manquant apporte au moins 0,2 point."
)
score_hypothese = essai(
    "solde_credit_manquant",
    lambda: modele.score(X.loc[Xval.index], yval),
    changement="Test de la feature solde_credit_manquant sur la validation fixe",
    attendu="Gain attendu d'au moins 0,2 point selon l'hypothèse."
)
gain_hypothese_points = (score_hypothese - score_interne) if score_hypothese is not None else 0.0
confirmee_ou_contredite = "confirmée" if gain_hypothese_points >= 0.2 else "contredite"
suite_donnee = (
    f"Le gain mesuré est de {gain_hypothese_points:.3f} point(s), pour un seuil annoncé de 0,2 point. "
    f"L'hypothèse est donc {confirmee_ou_contredite}. Je conserverai la feature seulement si son "
    f"gain se confirme sur un autre découpage et si la colonne solde_credit est disponible en production ; "
    f"sinon, je documenterai le résultat nul ou négatif au lieu de le cacher."
)
print(f"Gain mesuré pour solde_credit_manquant : {gain_hypothese_points:.3f} point(s)")
print("Verdict :", confirmee_ou_contredite)
print(suite_donnee)


,feature,ce qu'elle raconte,test,contredite si
0,solde_credit_manquant,Un solde de crédit manquant peut signaler un p...,Comparer le score de validation avec la featur...,L'hypothèse sera contredite si le gain est inf...


,feature,ce qu'elle raconte,test,contredite si
0,solde_credit_manquant,Un solde de crédit manquant peut signaler un p...,Comparer le score de validation avec la featur...,L'hypothèse sera contredite si le gain est inf...
1,rappel_a_delai_egal,"À délai égal, un rappel pourrait être associé ...",Comparer les taux d'absence avec et sans rappe...,L'hypothèse sera contredite si les tranches as...


,feature,ce qu'elle raconte,test,contredite si
0,solde_credit_manquant,Un solde de crédit manquant peut signaler un p...,Comparer le score de validation avec la featur...,L'hypothèse sera contredite si le gain est inf...
1,rappel_a_delai_egal,"À délai égal, un rappel pourrait être associé ...",Comparer les taux d'absence avec et sans rappe...,L'hypothèse sera contredite si les tranches as...
2,age_superieur_60_ans,Une tranche d'âge peut porter un signal d'orga...,"Créer un indicateur age_superieur_60, comparer...",L'hypothèse sera contredite si l'ajout de l'in...


,essai,attendu,raison,écrit à
0,rien faire du tout,c'est le minimum à battre,un modèle qui répond toujours la même chose va...,10:54:26
1,un modèle suffit-il ?,mieux que la référence,"les arbres captent une partie du signal, pas tout",10:54:26
2,retirer age,"Je prévois un score proche du modèle de base, ...","L'âge peut apporter de l'information, mais age...",10:54:48
3,max_iter 150,"Je prévois un changement modeste, éventuelleme...",Un plus grand nombre d'itérations peut amélior...,10:54:50
4,RandomForest 100 arbres,Je prévois une performance proche ou légèremen...,La forêt aléatoire est une autre famille d'arb...,10:54:51
5,solde_credit_manquant,"Je prévois un gain d'au moins 0,2 point, comme...",Vérifier sur la validation fixe si l'indicateu...,10:55:57


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"
2,retirer age,Donnée : retirer la colonne age,"Je prévois un score proche, car age_mois décri...",82.51,"oui, à 10:54:48"
3,max_iter 150,Réglage : max_iter=150 au lieu de la valeur pa...,Je prévois une petite amélioration si les arbr...,82.51,"oui, à 10:54:50"
4,RandomForest 100 arbres,Famille de modèle : RandomForestClassifier,"Je prévois un score proche, sans garantie d'am...",82.34,"oui, à 10:54:51"
5,solde_credit_manquant,Test de la feature solde_credit_manquant sur l...,"Gain attendu d'au moins 0,2 point selon l'hypo...",82.51,"oui, à 10:55:57"


Gain mesuré pour solde_credit_manquant : 0.000 point(s)
Verdict : contredite
Le gain mesuré est de 0.000 point(s), pour un seuil annoncé de 0,2 point. L'hypothèse est donc contredite. Je conserverai la feature seulement si son gain se confirme sur un autre découpage et si la colonne solde_credit est disponible en production ; sinon, je documenterai le résultat nul ou négatif au lieu de le cacher.


### Question 15 — La méthode hors de ce jeu de données : une table de crédit  ·  *carnet J2*  ·  *cœur*


La question que le cours doit rendre possible : **est-ce que la méthode survit ailleurs ?**

Voici l'en-tête d'une autre table — des demandes de crédit :

| colonne | ce qu'elle contient |
|---|---|
| `id_client` | l'identifiant du client |
| `date_defaut` | la date du défaut de paiement — renseignée **seulement** si l'emprunteur a fait défaut |
| `statut_6_mois` | « à jour / 30 jours / 90 jours », relevé **six mois après** l'octroi |
| `montant` | le montant demandé, en euros |
| `canal` | le canal de la demande : agence, web, téléphone |
| `score_interne` | le score de risque calculé **au moment de la demande** |

À faire : classez **chaque** colonne dans **une** des quatre familles de la séance 2, et
justifiez chacune en une phrase :
  · **identifiant** — elle désigne une ligne, elle ne la décrit pas ;
  · **connue après l'issue** — sa valeur n'existe pas encore au moment de décider ;
  · **risque de disparition** — elle existe ici et pourrait ne plus être alimentée en production ;
  · **utilisable** — disponible au moment de la décision, et elle décrit la ligne.

À écrire : `familles` (une famille par colonne) et `justifications` (une phrase par colonne).


In [30]:
# QUESTION 15 — Transférer la méthode à la table de crédit.
# Chaque colonne reçoit exactement une des quatre familles définies dans la consigne.
familles = {
    "id_client": "identifiant",
    "date_defaut": "connue après l'issue",
    "statut_6_mois": "connue après l'issue",
    "montant": "utilisable",
    "canal": "utilisable",
    "score_interne": "utilisable"
}

justifications = {
    "id_client": (
        "Cette valeur sert à reconnaître le client et non à décrire son risque ; elle doit être "
        "conservée pour relier les résultats, mais écartée des variables prédictives."
    ),
    "date_defaut": (
        "La date n'est renseignée que si le défaut survient ; elle dépend donc de l'issue future "
        "et ne peut pas être utilisée au moment de décider d'accorder le crédit."
    ),
    "statut_6_mois": (
        "Ce statut est mesuré six mois après l'octroi ; il n'existe pas encore lors de la décision "
        "initiale et constituerait une fuite de cible."
    ),
    "montant": (
        "Le montant demandé est connu au moment de la demande et décrit le dossier ; il peut donc "
        "être utilisé pour la prédiction, après vérification de sa qualité."
    ),
    "canal": (
        "Le canal de demande est connu lors de la demande et décrit la façon dont le dossier arrive ; "
        "il est utilisable si ce champ reste disponible en production."
    ),
    "score_interne": (
        "Ce score est calculé au moment de la demande et est disponible avant la décision ; il est "
        "utilisable à condition que son calcul ne dépende pas d'informations futures."
    )
}
print(pd.DataFrame({
    "colonne": list(familles.keys()),
    "famille": list(familles.values()),
    "justification": [justifications[c] for c in familles]
}))


         colonne               famille  \
0      id_client           identifiant   
1    date_defaut  connue après l'issue   
2  statut_6_mois  connue après l'issue   
3        montant            utilisable   
4          canal            utilisable   
5  score_interne            utilisable   

                                       justification  
0  Cette valeur sert à reconnaître le client et n...  
1  La date n'est renseignée que si le défaut surv...  
2  Ce statut est mesuré six mois après l'octroi ;...  
3  Le montant demandé est connu au moment de la d...  
4  Le canal de demande est connu lors de la deman...  
5  Ce score est calculé au moment de la demande e...  


## Les trois tableaux à rendre

C'est la pièce principale du devoir. Les fonctions `constat(...)`, `hypothese(...)` et
`noter(...)` sont définies dans la mise en place : appelez-les, elles affichent le tableau.

### 1. Cinq constats chiffrés

**Cinq** lignes, dont **deux au moins mesurées par vous** (les trois de la séance sont déjà
écrites). Chaque ligne a quatre morceaux : l'observation, le chiffre, **par rapport à quoi**, et
ce que vous en faites. Un constat sans comparaison ne compte pas.

### 2. Trois hypothèses de features

Une hypothèse est une **prédiction qu'on peut tester** : ce qu'elle raconte (avec le signe),
comment on la teste, et **ce qui la contredirait** — un résultat précis. La question 14 en teste
une sous son propre nom.

### 3. Le tableau d'essais

Une ligne par changement essayé. L'**attendu** porte un signe **et** une raison, et le **gain en
barres** décide du verdict : « réel » ou « dans le bruit ».

La cellule qui suit affiche les trois tableaux tels que vous les rendez.


In [31]:
print("CONSTATS CHIFFRÉS \n")
display(pd.DataFrame(CONSTATS))
print("HYPOTHÈSES DE FEATURES \n")
display(pd.DataFrame(HYPOTHESES))
display(pd.DataFrame(ESSAIS))
print("VOS CONTRÔLES \n")
display(pd.DataFrame(VERIFICATIONS))

# ── la relecture : des contrôles de FORME, qui ne révèlent aucune valeur.
print("LA RELECTURE \n")
verifier('votre identifiant GitHub est renseigné', GRAINE is not None,
         'les questions 2 et 11 mesurent sur un échantillon tiré de votre identifiant')
verifier('cinq constats', len(CONSTATS) == 5,
         'la question 3 en demande cinq — les trois de la séance plus deux des vôtres')
verifier('au moins deux constats mesurés par vous',
         sum(1 for c in CONSTATS if c['observation'] not in CONSTATS_DE_LA_SEANCE) >= 2,
         'les trois constats de la séance 1 ne comptent pas comme les vôtres')
verifier('chaque constat porte un chiffre ET une comparaison',
         all(c['chiffre'] and c['comparaison'] for c in CONSTATS),
         'un constat sans comparaison ne compte pas : « par rapport à quoi ? »')
verifier('trois hypothèses', len(HYPOTHESES) == 3,
         'la question 14 en demande trois')
verifier('chaque hypothèse dit ce qui la contredirait',
         all(str(h['contredite si']).strip() for h in HYPOTHESES),
         'une hypothèse que rien ne pourrait contredire ne vaut rien')
verifier('au moins trois essais à vous — les amorces ne comptent pas',
         sum(1 for e in ESSAIS if e['hypothèse'] not in AMORCES) >= 3,
         'la question 8 en demande trois, de vous')
verifier('chaque essai porte un attendu — un signe ET une raison',
         all(e['attendu'].strip() for e in ESSAIS),
         'jamais une valeur : un signe, et pourquoi')
verifier('les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle',
         all(e['prédiction écrite avant ?'] != '— aucune'
             for e in ESSAIS if e['hypothèse'] not in AMORCES),
         'si cette ligne tombe, quelqu un a modifié ESSAIS à la main : noter() le garantit')
verifier('une hypothèse a été testée sous son propre nom',
         any(e['hypothèse'] in [h['feature'] for h in HYPOTHESES] for e in ESSAIS),
         'la question 14 en teste une : le tableau d essais doit porter son nom')


# Recréer la soumission APRÈS les questions, avec le modèle final et la même préparation.
# Le fichier est écrit dans le répertoire de travail Kaggle, prêt à être téléchargé.
soumission = pd.DataFrame({
    ID: test[ID],
    CIBLE: predict(test).astype(int)
})
soumission.to_csv("submission.csv", index=False)
print("Soumission finale créée : submission.csv")
display(soumission.head())


CONSTATS CHIFFRÉS 



,observation,chiffre,comparaison,conséquence
0,une colonne ne dit rien,1 valeur(s) distincte(s),"sur les 149894 lignes de la table, soit une co...",l'écarter — et ce qu'on y gagne se mesure
1,deux colonnes disent la même chose,"corrélation 1,000 entre cout_acte et montant_acte","contre moins de 0,05 pour les autres paires su...",n'en garder qu'une
2,la cible est déséquilibrée,18.25 % d'absents,contre 50 % pour un problème équilibré : la ré...,"mesurer : cela conditionne la métrique, donc c..."
3,aucun identifiant ligne_id dupliqué,0 identifiant(s) dupliqué(s),sur 149894 lignes d'entraînement,"garder ligne_id pour relier les prédictions, m..."
4,des valeurs manquent dans solde_credit,8241 valeurs manquantes (5.50 %),"sur 149894 lignes, soit 5.50 % du jeu d'entraî...",garder solde_credit et mesurer l'effet d'un in...


HYPOTHÈSES DE FEATURES 



,feature,ce qu'elle raconte,test,contredite si
0,solde_credit_manquant,Un solde de crédit manquant peut signaler un p...,Comparer le score de validation avec la featur...,L'hypothèse sera contredite si le gain est inf...
1,rappel_a_delai_egal,"À délai égal, un rappel pourrait être associé ...",Comparer les taux d'absence avec et sans rappe...,L'hypothèse sera contredite si les tranches as...
2,age_superieur_60_ans,Une tranche d'âge peut porter un signal d'orga...,"Créer un indicateur age_superieur_60, comparer...",L'hypothèse sera contredite si l'ajout de l'in...


,hypothèse,changement,attendu,mesuré,prédiction écrite avant ?
0,rien faire du tout,classe majoritaire,c'est le minimum à battre,81.75,"oui, à 10:54:26"
1,un modèle suffit-il ?,HistGradientBoosting,mieux que la référence,82.51,"oui, à 10:54:26"
2,retirer age,Donnée : retirer la colonne age,"Je prévois un score proche, car age_mois décri...",82.51,"oui, à 10:54:48"
3,max_iter 150,Réglage : max_iter=150 au lieu de la valeur pa...,Je prévois une petite amélioration si les arbr...,82.51,"oui, à 10:54:50"
4,RandomForest 100 arbres,Famille de modèle : RandomForestClassifier,"Je prévois un score proche, sans garantie d'am...",82.34,"oui, à 10:54:51"
5,solde_credit_manquant,Test de la feature solde_credit_manquant sur l...,"Gain attendu d'au moins 0,2 point selon l'hypo...",82.51,"oui, à 10:55:57"


VOS CONTRÔLES 



""


LA RELECTURE 

  ok        votre identifiant GitHub est renseigné
  ok        cinq constats
  ok        au moins deux constats mesurés par vous
  ok        chaque constat porte un chiffre ET une comparaison
  ok        trois hypothèses
  ok        chaque hypothèse dit ce qui la contredirait
  ok        au moins trois essais à vous — les amorces ne comptent pas
  ok        chaque essai porte un attendu — un signe ET une raison
  ok        les essais À VOUS portent tous une prédiction écrite avant — état, pas contrôle
  ok        une hypothèse a été testée sous son propre nom
Soumission finale créée : submission.csv


,ligne_id,patient_absent
0,44088,0
1,120507,0
2,46612,0
3,3933,0
4,144527,0


**Ce qu'on rend, et pourquoi ce format.** Ce qu'on rend n'est pas un carnet : c'est un **texte**.
« Le modèle fait 82 % » ne dit rien ; « la cible est déséquilibrée à 18 %, donc 82 % vaut à peine
mieux que la référence » déclenche une décision. Entre les deux, la différence n'est pas le
chiffre : c'est la comparaison — et c'est exactement ce que les trois tableaux demandent.


## Rendre le travail

1. **Exécutez tout** le carnet (`Run All`), pour que les tableaux soient remplis.
2. **Déposez** ce carnet dans votre dépôt GitHub personnel, **avec** votre déclaration d'usage de
   l'IA, dans le **même commit** :
   - téléchargez le modèle **`DECLARATION_modele.md`** depuis le **dépôt du cours** ;
   - recopiez-le dans votre dépôt sous le nom **`declaration_DM1.md`** ;
   - remplissez-le : les deux scores, ce que vous avez produit seul, ce que vous avez fait
     avec un assistant, et ce que vous ne sauriez pas expliquer.
3. Vérifiez que le lien de votre dépôt est bien à jour dans le formulaire de la classe.

> Un travail non déposé n'existe pas.
